Cell 1 (RUN)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Cell 2 (RUN)

In [2]:
!pip install spacy
!pip install sentence-transformers
!pip install transformers
!pip install datasets
!pip install torch

!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 118.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


Cell 3 (RUN)

In [3]:
import pandas as pd
import numpy as np
import re
import spacy

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy import sparse
import joblib
import os

Cell 4 (RUN)

In [4]:
nlp = spacy.load("en_core_web_sm")

print("spaCy model loaded")

spaCy model loaded


Cell 5: Loading all datasets

In [ ]:
people_df = pd.read_csv(
    "/content/drive/MyDrive/NLP/01_people.csv"
)

abilities_df = pd.read_csv(
    "/content/drive/MyDrive/NLP/02_abilities.csv"
)

education_df = pd.read_csv(
    "/content/drive/MyDrive/NLP/03_education.csv"
)

experience_df = pd.read_csv(
    "/content/drive/MyDrive/NLP/04_experience.csv"
)

person_skill_df = pd.read_csv(
    "/content/drive/MyDrive/NLP/05_person_skills.csv"
)

skill_df = pd.read_csv(
    "/content/drive/MyDrive/NLP/06_skills.csv"
)

resume2 = pd.read_csv(
    "/content/drive/MyDrive/NLP/Resume.csv"
)

job = pd.read_csv(
    "/content/drive/MyDrive/NLP/fake_job_postings.csv"
)

esco = pd.read_csv(
    "/content/drive/MyDrive/NLP/skills.csv"
)

Cell 6

In [ ]:
abilities_grouped = (
    abilities_df
    .groupby("person_id")["ability"]
    .apply(", ".join)
    .reset_index()
)

merged_df = pd.merge(
    people_df,
    abilities_grouped,
    on="person_id",
    how="inner"
)

merged_df.to_csv(
    "/content/drive/MyDrive/NLP/merge_resume.csv",
    index=False
)

print(merged_df.head())

   person_id                                               name email phone  \
0          1                             Database Administrator   NaN   NaN   
1          2                             Database Administrator   NaN   NaN   
2          3                      Oracle Database Administrator   NaN   NaN   
3          4  Amazon Redshift Administrator and ETL Develope...   NaN   NaN   
4          5             Scrum Master Scrum Master Scrum Master   NaN   NaN   

  linkedin                                            ability  
0      NaN  Installation and Building Server, Running Back...  
1      NaN  database management systems administration, de...  
2      NaN  Over 4+ years of Experience as Architecture, E...  
3      NaN  SQL management, PostgresSQL, Oracle, MySQL, mi...  
4      NaN  Scrum Master, Agile software development, Prod...  


Cell 7: Merged resume 1

In [ ]:
resume1 = pd.read_csv(
    "/content/drive/MyDrive/NLP/merge_resume.csv"
)

print(resume1.shape)
print(resume2.shape)
print(job.shape)
print(esco.shape)

(54930, 6)
(3474, 169)
(17880, 18)
(13893, 2)


Cell 8: cleaning resume dataset (RUN)

In [5]:
def clean_text(text):

    text = str(text).lower()

    text = re.sub(r'\n', ' ', text)

    text = re.sub(
        r'[^a-zA-Z0-9+.# ]',
        ' ',
        text
    )

    text = re.sub(
        r'\s+',
        ' ',
        text
    ).strip()

    return text

Cell 9

In [ ]:
resume1 = resume1[
    ['name', 'ability']
]

resume1 = resume1.dropna()

resume1 = resume1.rename(
    columns={
        'name': 'category',
        'ability': 'resume_text'
    }
)

resume1['resume_text'] = (
    resume1['resume_text']
    .apply(clean_text)
)

resume1['category'] = (
    resume1['category']
    .apply(clean_text)
)

print(resume1.head())

                                            category  \
0                             database administrator   
1                             database administrator   
2                      oracle database administrator   
3  amazon redshift administrator and etl develope...   
4             scrum master scrum master scrum master   

                                         resume_text  
0  installation and building server running backu...  
1  database management systems administration dev...  
2  over 4+ years of experience as architecture ex...  
3  sql management postgressql oracle mysql micros...  
4  scrum master agile software development produc...  


Cell 10

In [ ]:
resume2 = resume2[
    ['Resume_str', 'Category']
]

resume2 = resume2.dropna()

resume2 = resume2.rename(
    columns={
        'Resume_str': 'resume_text',
        'Category': 'category'
    }
)

resume2['resume_text'] = (
    resume2['category'].astype(str)
    + " " +
    resume2['resume_text'].astype(str)
)

resume2['resume_text'] = (
    resume2['resume_text']
    .apply(clean_text)
)

resume2['category'] = (
    resume2['category']
    .apply(clean_text)
)

print(resume2.head())

                                         resume_text category
0  hr hr administrator marketing associate hr adm...       hr
1  hr hr specialist us hr operations summary vers...       hr
2  hr hr director summary over 20 years experienc...       hr
3  hr hr specialist summary dedicated driven and ...       hr
4  hr hr manager skill highlights hr skills hr de...       hr


Cell 11

In [ ]:
all_resumes = pd.concat(
    [resume1, resume2],
    ignore_index=True
)

print(all_resumes.shape)

print(all_resumes.head())

(57351, 2)
                                            category  \
0                             database administrator   
1                             database administrator   
2                      oracle database administrator   
3  amazon redshift administrator and etl develope...   
4             scrum master scrum master scrum master   

                                         resume_text  
0  installation and building server running backu...  
1  database management systems administration dev...  
2  over 4+ years of experience as architecture ex...  
3  sql management postgressql oracle mysql micros...  
4  scrum master agile software development produc...  


Cell 12

In [ ]:
job = job[
    ['title', 'description', 'requirements']
]

job = job.fillna('')

job['job_text'] = (
    job['title'].astype(str)
    + " " +
    job['description'].astype(str)
    + " " +
    job['requirements'].astype(str)
)

job['job_text'] = (
    job['job_text']
    .apply(clean_text)
)

print(job.head())

                                       title  \
0                           Marketing Intern   
1  Customer Service - Cloud Video Production   
2    Commissioning Machinery Assistant (CMA)   
3          Account Executive - Washington DC   
4                        Bill Review Manager   

                                         description  \
0  Food52, a fast-growing, James Beard Award-winn...   
1  Organised - Focused - Vibrant - Awesome!Do you...   
2  Our client, located in Houston, is actively se...   
3  THE COMPANY: ESRI – Environmental Systems Rese...   
4  JOB TITLE: Itemization Review ManagerLOCATION:...   

                                        requirements  \
0  Experience with content management systems a m...   
1  What we expect from you:Your key responsibilit...   
2  Implement pre-commissioning and commissioning ...   
3  EDUCATION: Bachelor’s or Master’s in GIS, busi...   
4  QUALIFICATIONS:RN license in the State of Texa...   

                                     

Cell 13

In [ ]:
print(esco.columns)

Index(['label_cleaned', 'altLabels'], dtype='object')


Cell 14

In [ ]:
esco_skills = (
    esco['label_cleaned']
    .dropna()
    .astype(str)
    .apply(clean_text)
    .tolist()
)

print("Total ESCO skills:", len(esco_skills))

Total ESCO skills: 13893


Cell 15

In [ ]:
from spacy.matcher import PhraseMatcher

esco_skills_clean = list(
    set([
        skill for skill in esco_skills
        if len(skill.split()) <= 4 and len(skill) >= 3
    ])
)

skill_matcher = PhraseMatcher(
    nlp.vocab,
    attr="LOWER"
)

patterns = [
    nlp.make_doc(skill)
    for skill in esco_skills_clean
]

skill_matcher.add(
    "SKILLS",
    patterns
)

def extract_entities_v2(text):
    doc = nlp(text)

    skills = []
    organizations = []
    education = []
    languages = []

    for ent in doc.ents:
        if ent.label_ == "ORG":
            organizations.append(ent.text)
        elif ent.label_ == "LANGUAGE":
            languages.append(ent.text)

    matches = skill_matcher(doc)

    for match_id, start, end in matches:
        skill = doc[start:end].text.lower().strip()
        skills.append(skill)

    education_keywords = [
        "bachelor",
        "master",
        "phd",
        "bsc",
        "msc",
        "computer science",
        "engineering",
        "university",
        "college",
        "hsc",
        "ssc",
        "gpa"
    ]

    for sent in doc.sents:
        sent_text = clean_text(sent.text)

        if any(word in sent_text for word in education_keywords):
            education.append(sent_text)

    return {
        "skills": sorted(list(set(skills))),
        "organizations": sorted(list(set(organizations))),
        "education": sorted(list(set(education))),
        "languages": sorted(list(set(languages)))
    }

No need to run again

1. NER Extraction

Cell 1

In [ ]:
print("Extracting job entities extractor...")

job['entities'] = (
    job['job_text']
    .apply(extract_entities_v2)
)


Extracting job entities extractor...


cell 2

In [ ]:
job.to_pickle(
    "/content/drive/MyDrive/NLP/job_entities_v2.pkl"
)

cell 3

In [ ]:
print("Extracting resume entities...")

all_resumes['entities'] = (
    all_resumes['resume_text']
    .apply(extract_entities)
)

Extracting resume entities...


cell 4

In [ ]:
all_resumes.to_pickle(
    "/content/drive/MyDrive/NLP/all_resumes_entities.pkl"
)

2. Model Training starts here

Cell 1

In [ ]:
sample_resumes = all_resumes.sample(
    5000,
    random_state=42
).reset_index(drop=True)

sample_jobs = job.sample(
    2000,
    random_state=42
).reset_index(drop=True)

print(sample_resumes.shape)
print(sample_jobs.shape)

(5000, 2)
(2000, 4)


Model 1: RoBERTa Large v1 model

Cell 2 (RUN)

In [6]:
roberta_model = SentenceTransformer(
    'sentence-transformers/all-roberta-large-v1'
)

print("RoBERTa loaded")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/9.68k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

RoBERTa loaded


Cell 3

In [ ]:
roberta_resume_embeddings = roberta_model.encode(
    all_resumes['resume_text'].tolist(),
    batch_size=16,
    show_progress_bar=True
)
np.save(
    "/content/drive/MyDrive/NLP/roberta_resume_embeddings.npy",
    roberta_resume_embeddings
)

Batches:   0%|          | 0/3585 [00:00<?, ?it/s]

Cell 4

In [ ]:
roberta_job_embeddings = roberta_model.encode(
    job['job_text'].tolist(),
    batch_size=16,
    show_progress_bar=True
)
np.save(
    "/content/drive/MyDrive/NLP/roberta_job_embeddings.npy",
    roberta_job_embeddings
)

Batches:   0%|          | 0/1118 [00:00<?, ?it/s]

Model 2: BERT All-MiniLM-L6-v2

Cell 5 (RUN)

In [7]:
bert_model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

print("Sentence-BERT loaded")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sentence-BERT loaded


Cell 6

In [ ]:
bert_resume_embeddings = bert_model.encode(
    all_resumes['resume_text'].tolist(),
    show_progress_bar=True
)
np.save(
    "/content/drive/MyDrive/NLP/bert_resume_embeddings.npy",
    bert_resume_embeddings
)

Batches:   0%|          | 0/1793 [00:00<?, ?it/s]

Cell 7

In [ ]:
bert_job_embeddings = bert_model.encode(
    job['job_text'].tolist(),
    show_progress_bar=True
)
np.save(
    "/content/drive/MyDrive/NLP/bert_job_embeddings.npy",
    bert_job_embeddings
)

Batches:   0%|          | 0/559 [00:00<?, ?it/s]

Model 3: All-distil-RoBERTa-v1 model

Cell 8 (RUN)

In [8]:
roberta_model2 = SentenceTransformer(
    'sentence-transformers/all-distilroberta-v1'
)

print("DistilRoBERTa loaded")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.1k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/653 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  328MB            

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

DistilRoBERTa loaded


Cell 9

In [ ]:
roberta_resume_embeddings2 = roberta_model2.encode(
    all_resumes['resume_text'].tolist(),
    batch_size=16,
    show_progress_bar=True
)
np.save(
    "/content/drive/MyDrive/NLP/roberta_resume_embeddings2.npy",
    roberta_resume_embeddings2
)

Batches:   0%|          | 0/3585 [00:00<?, ?it/s]

Cell 10

In [ ]:
roberta_job_embeddings2 = roberta_model2.encode(
    job['job_text'].tolist(),
    batch_size=16,
    show_progress_bar=True
)
np.save(
    "/content/drive/MyDrive/NLP/roberta_job_embeddings2.npy",
    roberta_job_embeddings2
)

Run from here again

Cell 16 (RUN)

Extract resume and job load korar jonno

In [9]:
all_resumes = pd.read_pickle(
 "/content/drive/MyDrive/NLP/all_resumes_entities.pkl"
)

job = pd.read_pickle(
  "/content/drive/MyDrive/NLP/job_entities.pkl"
)

Cell 17

In [ ]:
print(
    all_resumes[
        ['resume_text', 'entities']
    ].head()
)

                                         resume_text  \
0  installation and building server running backu...   
1  database management systems administration dev...   
2  over 4+ years of experience as architecture ex...   
3  sql management postgressql oracle mysql micros...   
4  scrum master agile software development produc...   

                                            entities  
0  {'skills': ['integrate building requirements i...  
1  {'skills': ['give instructions to staff', 'int...  
2  {'skills': ['develop gambling games', 'types o...  
3  {'skills': ['mysql', 'oracle relational databa...  
4  {'skills': ['perform smart grid feasibility st...  


Load all model embeddings

Cell 18

In [ ]:
bert_resume_embeddings = np.load(
    "/content/drive/MyDrive/NLP/bert_resume_embeddings.npy"
)

bert_job_embeddings = np.load(
    "/content/drive/MyDrive/NLP/bert_job_embeddings.npy"
)

roberta_resume_embeddings = np.load(
    "/content/drive/MyDrive/NLP/roberta_resume_embeddings.npy"
)

roberta_job_embeddings = np.load(
    "/content/drive/MyDrive/NLP/roberta_job_embeddings.npy"
)

roberta_resume_embeddings2 = np.load(
    "/content/drive/MyDrive/NLP/roberta_resume_embeddings2.npy"
)

roberta_job_embeddings2 = np.load(
    "/content/drive/MyDrive/NLP/roberta_job_embeddings2.npy"
)

print("Embeddings loaded")

Embeddings loaded


Cell 19

In [ ]:
RoBERT_SIMILARITY_PATH = "/content/drive/MyDrive/NLP/robert_similarity_matrix.npy"

if os.path.exists(RoBERT_SIMILARITY_PATH):
    print("RoBERTa similarity matrix")

    Robert_similarity_matrix = np.load(
        RoBERT_SIMILARITY_PATH
    )

else:

    Robert_similarity_matrix = cosine_similarity(
        roberta_resume_embeddings,
        roberta_job_embeddings
    )

    np.save(
        RoBERT_SIMILARITY_PATH,
        Robert_similarity_matrix
    )

    print("RoBERTa similarity matrix saved to Drive.")

print(Robert_similarity_matrix.shape)

RoBERTa similarity matrix
(57351, 17880)


Cell 20

In [ ]:
BERT_SIMILARITY_PATH = "/content/drive/MyDrive/NLP/bert_similarity_matrix.npy"

if os.path.exists(BERT_SIMILARITY_PATH):
    print("BERT similarity matrix")

    bert_similarity_matrix = np.load(
        BERT_SIMILARITY_PATH
    )

else:

    bert_similarity_matrix = cosine_similarity(
        bert_resume_embeddings,
        bert_job_embeddings
    )

    np.save(
        BERT_SIMILARITY_PATH,
        bert_similarity_matrix
    )

    print("BERT similarity matrix saved to Drive.")

print(bert_similarity_matrix.shape)

BERT similarity matrix
(57351, 17880)


Cell 21

In [ ]:
roberta_similarity_matrix2 = cosine_similarity(
    roberta_resume_embeddings2,
    roberta_job_embeddings2
)

print(roberta_similarity_matrix2.shape)

(57351, 17880)


Cell 22

In [ ]:
def calculate_skill_score(
    resume_skills,
    job_skills
):

    resume_skills = set(resume_skills)

    job_skills = set(job_skills)

    if len(job_skills) == 0:
        return 0

    matched = len(
        resume_skills.intersection(job_skills)
    )

    return matched / len(job_skills)

Cell 23

In [ ]:
TFIDF_DIR = "/content/drive/MyDrive/NLP/tfidf_baseline"

os.makedirs(
    TFIDF_DIR,
    exist_ok=True
)

TFIDF_VECTORIZER_PATH = f"{TFIDF_DIR}/tfidf_vectorizer.joblib"
TFIDF_RESUME_MATRIX_PATH = f"{TFIDF_DIR}/tfidf_resume_matrix.npz"
TFIDF_JOB_MATRIX_PATH = f"{TFIDF_DIR}/tfidf_job_matrix.npz"

Cell 24: TF-IDF Training

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=20000
)

tfidf_vectorizer.fit(
    pd.concat(
        [
            all_resumes["resume_text"],
            job["job_text"]
        ],
        ignore_index=True
    )
)

tfidf_resume_matrix = tfidf_vectorizer.transform(
    all_resumes["resume_text"]
)

tfidf_job_matrix = tfidf_vectorizer.transform(
    job["job_text"]
)

joblib.dump(
    tfidf_vectorizer,
    TFIDF_VECTORIZER_PATH
)

sparse.save_npz(
    TFIDF_RESUME_MATRIX_PATH,
    tfidf_resume_matrix
)

sparse.save_npz(
    TFIDF_JOB_MATRIX_PATH,
    tfidf_job_matrix
)

print("TF-IDF baseline trained")

TF-IDF baseline trained


Cell 25 : TF-IDF Loading

In [ ]:
if not os.path.exists(TFIDF_VECTORIZER_PATH):
    raise FileNotFoundError(
        "TF-IDF vectorizer not found"
    )

if not os.path.exists(TFIDF_RESUME_MATRIX_PATH):
    raise FileNotFoundError(
        "TF-IDF resume matrix not found"
    )

if not os.path.exists(TFIDF_JOB_MATRIX_PATH):
    raise FileNotFoundError(
        "TF-IDF job matrix not found"
    )

tfidf_vectorizer = joblib.load(
    TFIDF_VECTORIZER_PATH
)

tfidf_resume_matrix = sparse.load_npz(
    TFIDF_RESUME_MATRIX_PATH
)

tfidf_job_matrix = sparse.load_npz(
    TFIDF_JOB_MATRIX_PATH
)

print("TF-IDF files loaded successfully.")
print("TF-IDF resume matrix:", tfidf_resume_matrix.shape)
print("TF-IDF job matrix:", tfidf_job_matrix.shape)

TF-IDF files loaded successfully.
TF-IDF resume matrix: (57351, 20000)
TF-IDF job matrix: (17880, 20000)


Cell 26:TF-IDF match result

In [ ]:
TFIDF_RESULTS_PATH = f"{TFIDF_DIR}/tfidf_model_comparison.csv"

if os.path.exists(TFIDF_RESULTS_PATH):

    tfidf_df = pd.read_csv(
        TFIDF_RESULTS_PATH
    )

else:

    tfidf_matches = []

    batch_size = 500

    for start in range(
        0,
        tfidf_resume_matrix.shape[0],
        batch_size
    ):
        end = min(
            start + batch_size,
            tfidf_resume_matrix.shape[0]
        )

        batch_similarity = cosine_similarity(
            tfidf_resume_matrix[start:end],
            tfidf_job_matrix
        )

        for local_i in range(batch_similarity.shape[0]):
            i = start + local_i

            best_job_idx = np.argmax(
                batch_similarity[local_i]
            )

            resume_entities = all_resumes.iloc[i]["entities"]
            job_entities = job.iloc[best_job_idx]["entities"]

            similarity_score = batch_similarity[local_i][best_job_idx]

            skill_score = calculate_skill_score(
                resume_entities["skills"],
                job_entities["skills"]
            )

            final_score = (
                0.85 * similarity_score
                + 0.15 * skill_score
            )

            tfidf_matches.append({
                "model": "TF-IDF Baseline",
                "category": all_resumes.iloc[i]["category"],
                "job_title": job.iloc[best_job_idx]["title"],
                "similarity_score": similarity_score,
                "skill_score": skill_score,
                "final_score": final_score
            })

    tfidf_df = pd.DataFrame(
        tfidf_matches
    )

    tfidf_df.to_csv(
        TFIDF_RESULTS_PATH,
        index=False
    )

    print("TF-IDF match results saved to Drive.")

print(tfidf_df.head())

             model                                           category  \
0  TF-IDF Baseline                             database administrator   
1  TF-IDF Baseline                             database administrator   
2  TF-IDF Baseline                      oracle database administrator   
3  TF-IDF Baseline  amazon redshift administrator and etl develope...   
4  TF-IDF Baseline             scrum master scrum master scrum master   

                                         job_title  similarity_score  \
0                                Database Engineer          0.400128   
1  P/T CONTRACT - MySQL Database Analyst (Kahoots)          0.215709   
2                    Senior Lead Oracle DBA w/ RAC          0.233402   
3              Data Specialist: Advanced Analytics          0.203445   
4                                       Oracle DBA          0.235122   

   skill_score  final_score  
0     0.285714     0.382966  
1     0.031903     0.188138  
2     0.516566     0.275877  
3     0.

Cell 27

In [ ]:
BERT_MATCHES_PATH = "/content/drive/MyDrive/NLP/bert_matches.csv"

if os.path.exists(BERT_MATCHES_PATH):
    print("Sentence-BERT match results")

    bert_df = pd.read_csv(
        BERT_MATCHES_PATH
    )

else:

    bert_matches = []

    for i in range(len(all_resumes)):

        best_job_idx = np.argmax(
            bert_similarity_matrix[i]
        )

        resume_entities = (
            all_resumes.iloc[i]['entities']
        )

        job_entities = (
            job.iloc[best_job_idx]['entities']
        )

        similarity_score = (
            bert_similarity_matrix[i][best_job_idx]
        )

        skill_score = calculate_skill_score(
            resume_entities['skills'],
            job_entities['skills']
        )

        final_score = (
            0.85 * similarity_score
            + 0.15 * skill_score
        )

        bert_matches.append({
            "model": "Sentence-BERT",
            "category": all_resumes.iloc[i]['category'],
            "job_title": job.iloc[best_job_idx]['title'],
            "similarity_score": similarity_score,
            "skill_score": skill_score,
            "final_score": final_score
        })

    bert_df = pd.DataFrame(
        bert_matches
    )

    bert_df.to_csv(
        BERT_MATCHES_PATH,
        index=False
    )

    print("Sentence-BERT match results saved to Drive.")

print(bert_df.head())

Sentence-BERT match results
           model                                           category  \
0  Sentence-BERT                             database administrator   
1  Sentence-BERT                             database administrator   
2  Sentence-BERT                      oracle database administrator   
3  Sentence-BERT  amazon redshift administrator and etl develope...   
4  Sentence-BERT             scrum master scrum master scrum master   

                                job_title  similarity_score  skill_score  \
0        Application Developer/SQL Server          0.703745     0.077544   
1  Database Administrator & BI Developer           0.765426     0.076736   
2           Senior Lead Oracle DBA w/ RAC          0.754386     0.516566   
3                  PHP Software Developer          0.667370     0.013067   
4                              Oracle DBA          0.651557     0.008032   

   final_score  
0     0.609815  
1     0.662122  
2     0.718713  
3     0.569225  
4  

Cell 28

In [ ]:
RoBERT_MATCHES_PATH = "/content/drive/MyDrive/NLP/Robert_matches.csv"

if os.path.exists(RoBERT_MATCHES_PATH):
    print("Sentence-RoBERT match results")

    roberta_df = pd.read_csv(
        RoBERT_MATCHES_PATH
    )

else:

    Robert_matches = []

    for i in range(len(all_resumes)):
        best_job_idx = np.argmax(
            Robert_similarity_matrix[i]
        )

        resume_entities = (
            all_resumes.iloc[i]['entities']
        )

        job_entities = (
            job.iloc[best_job_idx]['entities']
        )

        similarity_score = (
            Robert_similarity_matrix[i][best_job_idx]
        )

        skill_score = calculate_skill_score(
            resume_entities['skills'],
            job_entities['skills']
        )

        final_score = (
            0.85 * similarity_score
            + 0.15 * skill_score
        )

        Robert_matches.append({
            "model": "Sentence-RoBERT",
            "category": all_resumes.iloc[i]['category'],
            "job_title": job.iloc[best_job_idx]['title'],
            "similarity_score": similarity_score,
            "skill_score": skill_score,
            "final_score": final_score
        })

    roberta_df = pd.DataFrame(
        Robert_matches
    )

    roberta_df.to_csv(
        RoBERT_MATCHES_PATH,
        index=False
    )

    print("Sentence-RoBERT match results saved to Drive.")

print(roberta_df.head())

Sentence-RoBERT match results
             model                                           category  \
0  Sentence-RoBERT                             database administrator   
1  Sentence-RoBERT                             database administrator   
2  Sentence-RoBERT                      oracle database administrator   
3  Sentence-RoBERT  amazon redshift administrator and etl develope...   
4  Sentence-RoBERT             scrum master scrum master scrum master   

                                          job_title  similarity_score  \
0  SQL Server DBA Job opportunity at Barrington, IL          0.803260   
1                         DW Database Administrator          0.679404   
2                     Senior Lead Oracle DBA w/ RAC          0.677380   
3                                           DB2 DBA          0.691525   
4                                   Oracle APPS DBA          0.602962   

   skill_score  final_score  
0     0.120083     0.700783  
1     0.125413     0.596306  
2 

Cell 29

In [ ]:
distilroberta_matches = []

for i in range(len(all_resumes)):

    best_job_idx = np.argmax(
        roberta_similarity_matrix2[i]
    )

    resume_entities = (
        all_resumes.iloc[i]['entities']
    )

    job_entities = (
        job.iloc[best_job_idx]['entities']
    )

    similarity_score = (
        roberta_similarity_matrix2[i][best_job_idx]
    )

    skill_score = calculate_skill_score(
        resume_entities['skills'],
        job_entities['skills']
    )

    final_score = (
        0.85 * similarity_score
        + 0.15 * skill_score
    )

    distilroberta_matches.append({

        "model": "DistilRoBERTa",

        "category":
            all_resumes.iloc[i]['category'],

        "job_title":
            job.iloc[best_job_idx]['title'],

        "similarity_score":
            similarity_score,

        "skill_score":
            skill_score,

        "final_score":
            final_score
    })

distilroberta_df = pd.DataFrame(
    distilroberta_matches
)

print(distilroberta_df.head())

           model                                           category  \
0  DistilRoBERTa                             database administrator   
1  DistilRoBERTa                             database administrator   
2  DistilRoBERTa                      oracle database administrator   
3  DistilRoBERTa  amazon redshift administrator and etl develope...   
4  DistilRoBERTa             scrum master scrum master scrum master   

                             job_title  similarity_score  skill_score  \
0                 SQL Server Developer          0.776883     0.108597   
1               Database Administrator          0.745733     0.183206   
2        Senior Lead Oracle DBA w/ RAC          0.759479     0.516566   
3  Operational Database Administrator           0.636999     0.062780   
4            Technical Project Manager          0.630586     0.002635   

   final_score  
0     0.676640  
1     0.661354  
2     0.723042  
3     0.550866  
4     0.536393  


Cell 30

In [ ]:
comparison_df = pd.concat(
    [
        bert_df,
        roberta_df,
        distilroberta_df,
        tfidf_df
    ],
    ignore_index=True
)

print(comparison_df.head())

           model                                           category  \
0  Sentence-BERT                             database administrator   
1  Sentence-BERT                             database administrator   
2  Sentence-BERT                      oracle database administrator   
3  Sentence-BERT  amazon redshift administrator and etl develope...   
4  Sentence-BERT             scrum master scrum master scrum master   

                                job_title  similarity_score  skill_score  \
0        Application Developer/SQL Server          0.703745     0.077544   
1  Database Administrator & BI Developer           0.765426     0.076736   
2           Senior Lead Oracle DBA w/ RAC          0.754386     0.516566   
3                  PHP Software Developer          0.667370     0.013067   
4                              Oracle DBA          0.651557     0.008032   

   final_score  
0     0.609815  
1     0.662122  
2     0.718713  
3     0.569225  
4     0.555028  


Cell 31

In [ ]:
def classify(score):

    if score >= 0.75:
        return "Strong"

    elif score >= 0.50:
        return "Average"

    else:
        return "Weak"

Cell 32

In [ ]:
comparison_df['label'] = (
    comparison_df['final_score']
    .apply(classify)
)

print(
    comparison_df['label']
    .value_counts()
)

label
Average    128612
Weak        96154
Strong       4638
Name: count, dtype: int64


Cell 33

In [ ]:
model_comparison = comparison_df.groupby(
    'model'
)[
    ['similarity_score', 'skill_score', 'final_score']
].mean()

print(model_comparison)

                 similarity_score  skill_score  final_score
model                                                      
DistilRoBERTa            0.637702     0.153891     0.565130
Sentence-BERT            0.629919     0.162226     0.559765
Sentence-RoBERT          0.635289     0.183839     0.567572
TF-IDF Baseline          0.226322     0.164225     0.217007


Cell 34

In [ ]:
comparison_df.to_csv(
    "/content/drive/MyDrive/NLP/final_model_comparison.csv",
    index=False
)

print("Final comparison dataset saved")

Final comparison dataset saved


Cell 35: Model Evaluation

In [ ]:
model_score_evaluation = (
    comparison_df.groupby("model")[
        ["similarity_score", "skill_score", "final_score"]
    ]
    .mean()
    .reset_index()
)

model_score_evaluation["predicted_category"] = (
    model_score_evaluation["final_score"].apply(classify)
)

model_score_evaluation = (
    model_score_evaluation
    .sort_values("final_score", ascending=False)
    .reset_index(drop=True)
)

model_score_evaluation["rank"] = (
    model_score_evaluation.index + 1
)

print("\nMODEL SCORE EVALUATION")
print(model_score_evaluation.to_string(index=False))

best_model = model_score_evaluation.iloc[0]

print("\nBest Model:", best_model["model"])
print("Best Final Score:", round(best_model["final_score"], 4))
print("Predicted Category:", best_model["predicted_category"])

MODEL_SCORE_EVALUATION_PATH = (
    "/content/drive/MyDrive/NLP/model_score_evaluation.csv"
)

model_score_evaluation.to_csv(
    MODEL_SCORE_EVALUATION_PATH,
    index=False
)


MODEL SCORE EVALUATION
          model  similarity_score  skill_score  final_score predicted_category  rank
Sentence-RoBERT          0.635289     0.183839     0.567572            Average     1
  DistilRoBERTa          0.637702     0.153891     0.565130            Average     2
  Sentence-BERT          0.629919     0.162226     0.559765            Average     3
TF-IDF Baseline          0.226322     0.164225     0.217007               Weak     4

Best Model: Sentence-RoBERT
Best Final Score: 0.5676
Predicted Category: Average


Cell 36

In [ ]:
HUMAN_EVAL_TEMPLATE_PATH2 = (
    "/content/drive/MyDrive/NLP/human_evaluation_filled.csv"
)

human_eval_df = pd.read_csv(
    HUMAN_EVAL_TEMPLATE_PATH2
)

human_eval_df["human_label"] = pd.to_numeric(
    human_eval_df["human_label"],
    errors="coerce"
)

human_eval_df = human_eval_df.dropna(
    subset=["human_label"]
)

human_eval_df["human_label"] = (
    human_eval_df["human_label"].astype(int)
)

human_eval_df["predicted_label"] = (
    human_eval_df["final_score"] >= 0.50
).astype(int)

y_true = human_eval_df["human_label"]
y_pred = human_eval_df["predicted_label"]

accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0
)

print("\n Model EVALUATION")

print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1-score :", round(f1, 4))



 Model EVALUATION
Accuracy : 0.55
Precision: 0.6724
Recall   : 0.6
F1-score : 0.6341


Cell 37:Correct Output

In [ ]:
human_eval_df["result"] = (
    human_eval_df["human_label"] ==
    human_eval_df["predicted_label"]
)

print(
    human_eval_df[
        human_eval_df["result"] == True
    ][
        [
            "model",
            "category",
            "job_title",
            "similarity_score",
            "skill_score",
            "final_score",
            "human_label",
            "predicted_label"
        ]
    ]
    .head(5)
    .to_string(index=False)
)

          model                                category                                   job_title  similarity_score  skill_score  final_score  human_label  predicted_label
TF-IDF Baseline                          java developer                           Database Engineer          0.212380     0.233516     0.215550            0                0
Sentence-RoBERT        sr it specialist desktop support      IT Support Specialist & Build Engineer          0.692772     0.840501     0.714931            1                1
  DistilRoBERTa                 front end web developer AngularJS,HTML5 and CSS Front End Developer          0.763636     0.003501     0.649616            1                1
TF-IDF Baseline user experience designer uxd ued xd ixd                         DISPOSITION TRAINEE          0.133722     0.019231     0.116548            0                0
Sentence-RoBERT                     lead java developer                        JAVA/J2EE developer           0.668366     0.001314

Cell 38: Wrong Output

In [ ]:
print(
    human_eval_df[
        human_eval_df["result"] == False
    ][
        [
            "model",
            "category",
            "job_title",
            "similarity_score",
            "skill_score",
            "final_score",
            "human_label",
            "predicted_label"
        ]
    ]
    .head(5)
    .to_string(index=False)
)

          model                                  category               job_title  similarity_score  skill_score  final_score  human_label  predicted_label
TF-IDF Baseline      network infrastructure administrator     Lead Linux Engineer          0.200196     0.055351     0.178469            1                0
TF-IDF Baseline                 java pim integration lead Sr. Java J2EE Developer          0.231874     0.133333     0.217093            1                0
  DistilRoBERTa                             ignacio lopez             BA Recon GL          0.635729     0.171477     0.566091            0                1
TF-IDF Baseline                          business analyst        Business Analyst          0.242084     0.547573     0.287907            1                0
  Sentence-BERT senior database and systems administrator              Oracle DBA          0.572381     0.003546     0.487056            1                0


Cell 39

In [ ]:
HUMAN_EVAL_RESULT_PATH = (
    "/content/drive/MyDrive/NLP/human_evaluation_results.csv"
)

human_eval_df.to_csv(
    HUMAN_EVAL_RESULT_PATH,
    index=False
)

print("\nHuman evaluation results saved:")
print(HUMAN_EVAL_RESULT_PATH)


Human evaluation results saved:
/content/drive/MyDrive/NLP/human_evaluation_results.csv


Cell 40 (RUN)

In [10]:
!pip install PyMuPDF
!apt-get update -qq
!apt-get install -y tesseract-ocr tesseract-ocr-eng poppler-utils
!pip install pytesseract pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 80.1 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
tesseract-ocr-eng is already the newest version (1:4.00~git30-7274cfa-1.1).
tesseract-ocr-eng set to manually installed.
The following additional packages will be installed:
  libpoppler-dev libpoppler-private-dev libpoppler118
Recommended packages:
  poppler-data
The following NEW packages will be installed:
  poppler-utils
The following packages will be upgraded:
  libpoppler-dev libpoppler-private-dev libpoppler118
3 upgraded, 1 newly installed, 0 to remove and 170 not upgraded.
Need to get 1,471 kB of archives.
After this operation, 697 kB of additional disk spa

Cell 41 (RUN)

In [11]:
import fitz
import re
import io
import numpy as np
import pandas as pd
import pytesseract

from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from google.colab import files

Cell 42 (RUN)

In [12]:
print("Upload resume PDF")
resume_uploaded = files.upload()
resume_pdf_path = list(resume_uploaded.keys())[0]

print("Upload job description PDF")
job_uploaded = files.upload()
job_pdf_path = list(job_uploaded.keys())[0]

Upload resume PDF


Saving cv full SQA.pdf to cv full SQA.pdf
Upload job description PDF


Saving sqa engineer.pdf to sqa engineer.pdf


Cell 43 (RUN)

In [13]:
def extract_text_from_pdf_with_ocr(
    pdf_path,
    min_page_text_length=80,
    zoom=2,
    ocr_lang="eng"
):
    final_text = ""
    extraction_report = {
        "total_pages": 0,
        "text_pages": 0,
        "ocr_pages": 0,
        "failed_pages": 0
    }

    doc = fitz.open(pdf_path)
    extraction_report["total_pages"] = len(doc)

    for page in doc:
        page_text = page.get_text("text")

        if page_text is not None and len(page_text.strip()) >= min_page_text_length:
            final_text += page_text + "\n"
            extraction_report["text_pages"] += 1
        else:
            try:
                mat = fitz.Matrix(zoom, zoom)
                pix = page.get_pixmap(matrix=mat)

                img_bytes = pix.tobytes("png")
                image = Image.open(io.BytesIO(img_bytes))

                ocr_text = pytesseract.image_to_string(
                    image,
                    lang=ocr_lang
                )

                final_text += ocr_text + "\n"
                extraction_report["ocr_pages"] += 1

            except Exception:
                extraction_report["failed_pages"] += 1

    return final_text, extraction_report

Cell 44 (RUN)

In [14]:
def check_extracted_text_quality(text, extraction_report=None):
    text_clean = clean_text(text)
    word_count = len(text_clean.split())

    if word_count < 30:
        status = "Low text quality"
        message = "The PDF may be scanned, low-resolution, image-heavy, or difficult to OCR."
    elif word_count < 100:
        status = "Moderate text quality"
        message = "Text was extracted, but the content may be incomplete."
    else:
        status = "Good text quality"
        message = "Text extraction appears sufficient."

    result = {
        "status": status,
        "word_count": word_count,
        "message": message
    }

    if extraction_report is not None:
        result.update(extraction_report)

    return result

Cell 45 (RUN)

In [15]:
def get_text_from_pdf(pdf_path):
    text, report = extract_text_from_pdf_with_ocr(pdf_path)
    return text

def get_text_from_pdf_with_quality(pdf_path):
    text, report = extract_text_from_pdf_with_ocr(pdf_path)
    quality = check_extracted_text_quality(text, report)

    return text, quality

Cell 46 (RUN)

In [16]:
def get_cv_text(pdf_path=None, pasted_text=None):
    if pasted_text is not None and len(pasted_text.strip()) > 0:
        return pasted_text

    if pdf_path is not None:
        text, quality = get_text_from_pdf_with_quality(pdf_path)
        return text

    return ""

Cell 47 (RUN)

In [17]:
SECTION_KEYWORDS = {
    "education": [
        "education", "academic qualifications", "academic background",
        "qualification", "academic record"
    ],
    "experience": [
        "experience", "work experience", "employment history",
        "professional experience", "internship"
    ],
    "projects": [
        "projects", "academic projects", "personal projects",
        "ai ml and deep learning projects", "research projects"
    ],
    "skills": [
        "skills", "technical skills", "soft skills", "tools",
        "programming", "technologies"
    ],
    "certifications": [
        "certifications", "certificate", "courses", "training"
    ],
    "achievements": [
        "achievements", "awards", "honors", "leadership",
        "co-curricular activities", "extracurricular"
    ],
    "publications": [
        "publications", "research papers", "journal", "conference"
    ],
    "languages": [
        "languages", "language proficiency"
    ]
}


def extract_sections(text):
    lines = text.splitlines()

    sections = {
        key: []
        for key in SECTION_KEYWORDS.keys()
    }

    current_section = None

    for line in lines:
        line_clean = clean_text(line)

        matched_section = None

        for section, keywords in SECTION_KEYWORDS.items():
            for keyword in keywords:
                if keyword in line_clean and len(line_clean.split()) <= 8:
                    matched_section = section
                    break

            if matched_section:
                break

        if matched_section:
            current_section = matched_section
            continue

        if current_section:
            if len(line.strip()) > 0:
                sections[current_section].append(line.strip())

    return {
        section: " ".join(content)
        for section, content in sections.items()
    }

Cell 48 (RUN)

In [18]:
TECH_SKILLS = [
    # Programming
    "python", "java", "c++", "c", "c#", "javascript", "typescript",
    "html", "css", "php", "sql", "mysql", "postgresql", "mongodb",
    "dart",

    # AI / ML
    "machine learning", "deep learning", "artificial intelligence",
    "data science", "data analysis", "computer vision", "nlp",
    "natural language processing", "tensorflow", "keras", "pytorch",
    "scikit learn", "sklearn", "pandas", "numpy", "matplotlib",
    "seaborn", "opencv", "xgboost", "random forest", "svm",
    "logistic regression", "decision tree", "cnn", "mobilenet",
    "efficientnet", "bert", "roberta", "transformers",
    "feature engineering", "data preprocessing", "model evaluation",
    "classification", "regression",

    # Web / App / Backend
    "flask", "django", "fastapi", "streamlit", "gradio",
    "android", "flutter", "firebase", "react", "node", "express",
    "api", "rest api", "backend", "frontend", "database",

    # QA / Testing
    "qa", "quality assurance", "software testing", "manual testing",
    "automated testing", "automation testing", "test automation",
    "selenium", "pytest", "nunit", "unit testing", "integration testing",
    "regression testing", "test cases", "bug reporting", "defect tracking",
    "software quality", "quality control", "postman",

    # Tools / Platforms
    "git", "github", "docker", "kubernetes", "aws", "azure",
    "google cloud", "linux", "vs code", "android studio",
    "google colab", "overleaf"

    # Web/App/Backend
    "react native", "redux", "nodejs", "nestjs", "microservices",
    "microservice architecture", "rabbitmq", "kafka", "tdd",
    "test driven development", "web services", "sass", "less",
    "ubuntu", "debian", "monitoring", "observability",
    "application performance monitoring"

    #Auditor
    "ASPCA", "CSCA", "Auditor", "Lead auditor", "WRAP", "ISO"

    #Industry Type
    "Apparel", "Textiles", "Home Textile", "Footwear", "Plastic", "Jute items", "Accessories",
    "Leather", "Luggage", "Personal Care", "Sporting goods", "Art materials", "Eyewear", "Electronic industry",
    "Stationary items", "Sanitary items", "Household items"
]
ROLE_KEYWORDS = [
    "software engineer", "software developer", "backend developer",
    "frontend developer", "full stack developer", "web developer",
    "mobile app developer", "android developer", "flutter developer",
    "ai engineer", "machine learning engineer", "data analyst",
    "data scientist", "qa tester", "qa engineer", "automation engineer",
    "research assistant", "intern", "trainee"
]

DOMAIN_KEYWORDS = [
    "web development", "mobile application", "mobile development",
    "backend systems", "frontend development", "database management",
    "artificial intelligence", "machine learning", "deep learning",
    "computer vision", "cybersecurity", "intrusion detection",
    "automation testing", "software testing", "quality assurance",
    "data preprocessing", "model evaluation", "image classification",
    "risk prediction", "ensemble learning"
]

TOOL_KEYWORDS = [
    "vs code", "android studio", "google colab", "overleaf",
    "postman", "firebase", "github", "git", "selenium",
    "pytest", "nunit", "tkinter", "smote", "roc auc",
    "confusion matrix", "accuracy", "recall", "feature encoding",
    "scaling", "restful services", "real time database"
]

ACTION_KEYWORDS = [
    "developed", "built", "created", "implemented", "designed",
    "managed", "trained", "tested", "automated", "evaluated",
    "analyzed", "improved", "deployed", "integrated", "compared",
    "predicted", "classified", "optimized", "validated"
]

ACHIEVEMENT_KEYWORDS_EXTRA = [
    "winner", "gold medalist", "award", "competition",
    "president", "chair", "campus ambassador", "leadership",
    "student branch", "women in computing", "quiz club"
]

GENERAL_CV_KEYWORDS = sorted(
    list(
        set(
            ROLE_KEYWORDS
            + DOMAIN_KEYWORDS
            + TOOL_KEYWORDS
            + ACTION_KEYWORDS
            + ACHIEVEMENT_KEYWORDS_EXTRA
        )
    )
)
SOFT_SKILLS = [
    "communication", "teamwork", "leadership", "problem solving",
    "critical thinking", "time management", "adaptability",
    "collaboration", "creativity", "analytical thinking",
    "detail oriented", "fast learner"
]

SKILL_ALIASES = {
    "quality assurance": ["quality assurance", "qa"],
    "manual testing": ["manual testing", "manual tester"],
    "automated testing": ["automated testing", "automation testing"],
    "test automation": ["test automation", "automated test", "automated tests"],
    "software testing": ["software testing", "testing"],
    "software quality": ["software quality", "quality software"],
    "bug reporting": ["bug reporting", "bug reports", "report bugs"],
    "test cases": ["test cases", "test case"],
    "rest api": ["rest api", "rest apis", "restful services", "restful api"],
    "github": ["github", "git hub"],
    "detail oriented": ["detail oriented", "detail-oriented"],
    "c++": ["c++", "cpp"],
    "c#": ["c#", "c sharp"],
    "scikit learn": ["scikit learn", "scikit-learn", "sklearn"],
    "node": ["node", "nodejs", "node js"],
    "react": ["react", "reactjs", "react js"],
    "react native": ["react native", "react-native"],
    "rest api": ["rest api", "rest apis", "restful services", "restful api", "rest specifically"],
    "microservices": ["microservices", "microservice", "microservice architecture"],
    "tdd": ["tdd", "test driven development"],
    "web services": ["web services", "web service"],
    "application performance monitoring": ["application performance monitoring", "apm"],
    "industry type": ["Apparel", "Textiles", "Home Textile", "Footwear", "Plastic", "Jute items", "Accessories",
    "Leather", "Luggage", "Personal Care", "Sporting goods", "Art materials", "Eyewear", "Electronic industry",
    "Stationary items", "Sanitary items", "Household items"],
}

def extract_skills_from_text(text):
    text = clean_text(text)

    found_skills = set()

    all_skills = TECH_SKILLS + SOFT_SKILLS

    for skill in all_skills:
        aliases = SKILL_ALIASES.get(skill, [skill])

        for alias in aliases:
            alias_clean = clean_text(alias)

            # For skills with symbols like c++ and c#
            if alias in ["c++", "c#"]:
                if alias.lower() in str(text).lower():
                    found_skills.add(skill)
            else:
                pattern = r'(?<![a-zA-Z0-9])' + re.escape(alias_clean) + r'(?![a-zA-Z0-9])'

                if re.search(pattern, text):
                    found_skills.add(skill)

    return sorted(list(found_skills))

Cell 49 (RUN)

In [19]:
def extract_known_keywords_from_text(text):
    text_clean = clean_text(text)

    found_keywords = []

    for keyword in GENERAL_CV_KEYWORDS:
        keyword_clean = clean_text(keyword)

        pattern = (
            r'(?<![a-zA-Z0-9])'
            + re.escape(keyword_clean)
            + r'(?![a-zA-Z0-9])'
        )

        if re.search(pattern, text_clean):
            found_keywords.append(keyword)

    return sorted(list(set(found_keywords)))

Cell 50 (RUN)

In [20]:
def extract_frequent_keywords_from_text(text, top_n=30):
    text_clean = clean_text(text)

    custom_stop_words = [
        "name", "father", "mother", "date", "birth",
        "address", "phone", "email", "dhaka", "bd",
        "cv", "resume", "linkedin", "github"
    ]

    vectorizer = CountVectorizer(
        stop_words="english",
        ngram_range=(1, 3),
        max_features=1000,
        token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z0-9+#.-]{1,}\b'
    )

    try:
        keyword_matrix = vectorizer.fit_transform([text_clean])

        keywords = vectorizer.get_feature_names_out()
        scores = keyword_matrix.toarray()[0]

        keyword_scores = []

        for keyword, score in zip(keywords, scores):
            if score > 0:
                if keyword not in custom_stop_words:
                    if len(keyword) >= 3:
                        keyword_scores.append(
                            (keyword, score)
                        )

        keyword_scores = sorted(
            keyword_scores,
            key=lambda item: item[1],
            reverse=True
        )

        top_keywords = [
            keyword
            for keyword, score in keyword_scores[:top_n]
        ]

        return top_keywords

    except Exception:
        return []
def extract_cv_keywords(text):
    known_keywords = extract_known_keywords_from_text(text)

    frequent_keywords = extract_frequent_keywords_from_text(
        text,
        top_n=30
    )

    return {
        "known_keywords": known_keywords,
        "frequent_keywords": frequent_keywords,
        "keyword_count": len(set(known_keywords + frequent_keywords))
    }

Cell 51 (RUN)

In [21]:
LANGUAGE_LIST = [
    "english", "bangla", "bengali", "hindi", "urdu", "arabic",
    "french", "spanish", "german", "chinese", "japanese", "korean"
]


def extract_education_features(text, sections):
    education_text = clean_text(sections.get("education", "") + " " + text)

    degrees = []

    degree_keywords = [
        "bachelor", "bsc", "b.sc", "master", "msc", "m.sc",
        "phd", "diploma", "hsc", "ssc", "higher secondary",
        "secondary school"
    ]

    for degree in degree_keywords:
        if degree in education_text:
            degrees.append(degree)

    gpa_matches = re.findall(
        r'gpa\s*[:\-]?\s*([0-9]\.?[0-9]*)\s*/?\s*([0-9]\.?[0-9]*)?',
        education_text
    )

    institutions = []

    institution_keywords = [
        "university", "college", "school", "institute"
    ]

    sentences = re.split(r'[.\n]', text)

    for sent in sentences:
        sent_clean = clean_text(sent)

        if any(word in sent_clean for word in institution_keywords):
            institutions.append(sent.strip())

    return {
        "degrees": sorted(list(set(degrees))),
        "gpa_count": len(gpa_matches),
        "institutions": sorted(list(set(institutions)))[:5]
    }


def extract_experience_features(text, sections):
    possible_experience_text = " ".join([
        sections.get("experience", ""),
        sections.get("projects", ""),
        text
    ])

    experience_text = clean_text(possible_experience_text)

    year_patterns = [
        r'(\d+)\+?\s*(?:years|year|yrs|yr)\s+(?:of\s+)?(?:professional\s+)?experience',
        r'(\d+)\s*[-to]+\s*(\d+)\s*(?:years|year|yrs|yr)',
        r'(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\s+\d{4}\s*[-to]+\s*(?:present|current|now|\d{4})',
        r'\d{4}\s*[-to]+\s*(?:present|current|now|\d{4})'
    ]

    extracted_years = []

    for pattern in year_patterns:
        matches = re.findall(
            pattern,
            experience_text
        )

        for match in matches:
            if isinstance(match, tuple):
                numbers = [
                    int(value)
                    for value in match
                    if str(value).isdigit()
                ]

                if len(numbers) == 2:
                    extracted_years.append(
                        max(numbers) - min(numbers)
                    )
                elif len(numbers) == 1:
                    extracted_years.append(
                        numbers[0]
                    )
            else:
                if str(match).isdigit():
                    extracted_years.append(
                        int(match)
                    )

    experience_keywords = [
        "work experience", "professional experience", "employment",
        "career history", "work history", "internship", "intern",
        "trainee", "developer", "engineer", "officer", "assistant",
        "executive", "manager", "company", "organization",
        "responsibilities", "worked", "employed", "freelance",
        "contract", "client", "remote", "onsite"
    ]

    has_work_experience = any(
        keyword in experience_text
        for keyword in experience_keywords
    )

    estimated_years = 0

    if len(extracted_years) > 0:
        estimated_years = max(extracted_years)

    return {
        "estimated_years_experience": estimated_years,
        "has_work_experience": has_work_experience,
        "fresher_project_compensation": False
    }


def extract_project_features(text, sections):
    project_text = sections.get("projects", "")

    project_lines = [
        line.strip()
        for line in project_text.split(".")
        if len(line.strip()) > 20
    ]

    project_keywords = [
        "developed", "built", "created", "implemented",
        "designed", "prediction", "classification",
        "application", "system", "website", "framework"
    ]

    project_count = 0

    for line in project_lines:
        line_clean = clean_text(line)

        if any(keyword in line_clean for keyword in project_keywords):
            project_count += 1

    if project_count == 0:
        full_text = clean_text(text)

        project_count = len(
            re.findall(
                r'\b(developed|built|created|implemented|designed)\b',
                full_text
            )
        )

    return {
        "project_count": project_count,
        "project_keywords_found": sorted(
            list(
                set([
                    keyword
                    for keyword in project_keywords
                    if keyword in clean_text(text)
                ])
            )
        )
    }


def add_fresher_project_compensation(experience, projects):
    experience["fresher_project_compensation"] = (
        not experience["has_work_experience"]
        and projects["project_count"] >= 3
    )

    return experience


def extract_language_features(text, sections):
    language_section = clean_text(sections.get("languages", ""))

    languages = []

    for language in LANGUAGE_LIST:
        pattern = r'(?<![a-zA-Z0-9])' + re.escape(language) + r'(?![a-zA-Z0-9])'

        if re.search(pattern, language_section):
            languages.append(language)

    return {
        "languages": sorted(list(set(languages))),
        "language_count": len(set(languages))
    }


def extract_certification_features(text, sections):
    cert_text = clean_text(sections.get("certifications", ""))

    if len(cert_text.strip()) == 0:
        return {
            "has_certifications": False,
            "certification_keywords": []
        }

    certification_keywords = [
        "certification", "certified", "course",
        "training", "workshop", "bootcamp",
        "coursera", "udemy", "edx", "kaggle",
        "google certificate", "aws certified",
        "microsoft certified"
    ]

    found = [
        keyword
        for keyword in certification_keywords
        if keyword in cert_text
    ]

    return {
        "has_certifications": len(found) > 0,
        "certification_keywords": sorted(list(set(found)))
    }


def extract_achievement_features(text, sections):
    achievement_text = clean_text(
        sections.get("achievements", "") + " " + text
    )

    achievement_keywords = [
        "winner", "award", "gold medal", "medalist",
        "competition", "president", "chair", "leader",
        "leadership", "ambassador", "volunteer"
    ]

    found = [
        keyword
        for keyword in achievement_keywords
        if keyword in achievement_text
    ]

    return {
        "has_achievements": len(found) > 0,
        "achievement_keywords": sorted(list(set(found))),
        "leadership_count": len(
            re.findall(
                r'\b(president|chair|leader|leadership|ambassador)\b',
                achievement_text
            )
        )
    }


def extract_publication_features(text, sections):
    publication_text = clean_text(sections.get("publications", ""))

    if len(publication_text.strip()) == 0:
        return {
            "has_publications": False,
            "publication_keywords": []
        }

    publication_keywords = [
        "publication", "published", "journal",
        "conference paper", "research paper",
        "doi", "springer", "elsevier", "acm",
        "scopus", "indexed"
    ]

    found = [
        keyword
        for keyword in publication_keywords
        if keyword in publication_text
    ]

    return {
        "has_publications": len(found) > 0,
        "publication_keywords": sorted(list(set(found)))
    }

Cell 52 (RUN)

In [22]:
def create_structured_cv_features(cv_text):
    sections = extract_sections(cv_text)

    cv_clean = clean_text(cv_text)

    technical_and_soft_skills = extract_skills_from_text(cv_clean)

    education = extract_education_features(cv_text, sections)
    experience = extract_experience_features(cv_text, sections)
    projects = extract_project_features(cv_text, sections)

    experience = add_fresher_project_compensation(
        experience,
        projects
    )

    languages = extract_language_features(cv_text, sections)
    certifications = extract_certification_features(cv_text, sections)
    achievements = extract_achievement_features(cv_text, sections)
    publications = extract_publication_features(cv_text, sections)

    cv_keywords = extract_cv_keywords(cv_text)

    soft_skills_found = sorted(
        list(set(technical_and_soft_skills).intersection(set(SOFT_SKILLS)))
    )

    technical_skills_found = sorted(
        list(set(technical_and_soft_skills).intersection(set(TECH_SKILLS)))
    )

    features = {
        "technical_skills": technical_skills_found,
        "technical_skill_count": len(technical_skills_found),

        "soft_skills": soft_skills_found,
        "soft_skill_count": len(soft_skills_found),

        "keywords": cv_keywords,

        "education": education,
        "experience": experience,
        "projects": projects,
        "languages": languages,
        "certifications": certifications,
        "achievements": achievements,
        "publications": publications
    }

    return features

Cell 53 (RUN)

In [23]:
def calculate_rubric_score(features):
    score = 0
    breakdown = {}
    education = features["education"]

    academic_score = 0

    if len(education["degrees"]) > 0:
        academic_score += 8

    if education["gpa_count"] > 0:
        academic_score += 4

    if len(education["institutions"]) > 0:
        academic_score += 3

    academic_score = min(academic_score, 15)
    breakdown["academic_record"] = academic_score
    score += academic_score
    experience = features["experience"]
    project_count = features["projects"]["project_count"]

    experience_score = 0

    if experience["has_work_experience"]:
        experience_score += 7

    if experience["estimated_years_experience"] >= 1:
        experience_score += 4

    if experience["estimated_years_experience"] >= 2:
        experience_score += 4

    if experience["fresher_project_compensation"]:
        if project_count >= 3:
            experience_score += 5

        if project_count >= 5:
            experience_score += 3

    experience_score = min(experience_score, 15)
    breakdown["work_experience"] = experience_score
    score += experience_score

    project_count = features["projects"]["project_count"]

    project_score = 0

    if project_count >= 1:
        project_score += 5

    if project_count >= 3:
        project_score += 5

    if project_count >= 5:
        project_score += 5

    project_score = min(project_score, 15)
    breakdown["project_experience"] = project_score
    score += project_score

    tech_count = features["technical_skill_count"]

    technical_score = 0

    if tech_count >= 3:
        technical_score += 5

    if tech_count >= 8:
        technical_score += 7

    if tech_count >= 15:
        technical_score += 8

    technical_score = min(technical_score, 20)
    breakdown["technical_skills"] = technical_score
    score += technical_score

    # Soft skills: 10
    soft_count = features["soft_skill_count"]

    soft_score = 0

    if soft_count >= 1:
        soft_score += 4

    if soft_count >= 3:
        soft_score += 3

    if soft_count >= 5:
        soft_score += 3

    soft_score = min(soft_score, 10)
    breakdown["soft_skills"] = soft_score
    score += soft_score

    # Language proficiency: 5
    language_score = 0

    if features["languages"]["language_count"] >= 1:
        language_score += 3

    if features["languages"]["language_count"] >= 2:
        language_score += 2

    language_score = min(language_score, 5)
    breakdown["language_proficiency"] = language_score
    score += language_score

    # Certifications: 5
    certification_score = 5 if features["certifications"]["has_certifications"] else 0
    breakdown["certifications"] = certification_score
    score += certification_score

    # Achievements and leadership: 10
    achievement_score = 0

    if features["achievements"]["has_achievements"]:
        achievement_score += 5

    if features["achievements"]["leadership_count"] >= 1:
        achievement_score += 5

    achievement_score = min(achievement_score, 10)
    breakdown["achievements_leadership"] = achievement_score
    score += achievement_score

    # Publications: 5
    publication_score = 5 if features["publications"]["has_publications"] else 0
    breakdown["publications"] = publication_score
    score += publication_score

    return {
        "rubric_score": score,
        "breakdown": breakdown
    }

Cell 54 (RUN)

In [24]:
def classify_cv_from_100(score):
    if score >= 75:
        return "Strong"
    elif score >= 50:
        return "Average"
    else:
        return "Weak"

Cell 55 (RUN)

In [25]:
def suggest_improvements_from_features(features, rubric_result):
    suggestions = []

    breakdown = rubric_result["breakdown"]

    if breakdown["work_experience"] < 8:
        suggestions.append(
            "Add internship, freelance work, part-time work, or real client/project experience if available."
        )

    if breakdown["certifications"] == 0:
        suggestions.append(
            "Add relevant certifications or online courses such as machine learning, or Python."
        )

    if breakdown["language_proficiency"] == 0:
        suggestions.append(
            "Add a language proficiency section, for example English and Bangla."
        )

    if breakdown["publications"] == 0:
        suggestions.append(
            "Add publications only if you have research papers. Otherwise, this section can be skipped."
        )

    if breakdown["project_experience"] < 10:
        suggestions.append(
            "Add more project details with tools, methods, datasets, results, and your personal contribution."
        )

    if breakdown["technical_skills"] < 15:
        suggestions.append(
            "Add more relevant technical skills and tools based on the target job."
        )

    if len(suggestions) == 0:
        suggestions.append(
            "The CV is well structured and contains strong information across most rubric categories."
        )

    return suggestions

Cell 56 (RUN)

In [26]:
def evaluate_cv_without_job(cv_text):
    features = create_structured_cv_features(cv_text)

    rubric_result = calculate_rubric_score(features)

    cv_category = classify_cv_from_100(
        rubric_result["rubric_score"]
    )

    suggestions = suggest_improvements_from_features(
        features,
        rubric_result
    )

    result = {
        "rubric_score": rubric_result["rubric_score"],
        "cv_category": cv_category,
        "score_breakdown": rubric_result["breakdown"],
        "structured_features": features,
        "suggestions": suggestions
    }

    return result

Cell 57 (RUN)

In [27]:
def extract_required_experience_years(job_text):
    job_clean = clean_text(job_text)

    experience_patterns = [
        r'(\d+)\+?\s*(?:years|year|yrs|yr)\s+(?:of\s+)?(?:professional\s+)?experience',
        r'(?:minimum|min|at least)\s+(\d+)\+?\s*(?:years|year|yrs|yr)',
        r'(\d+)\s*[-to]+\s*(\d+)\s*(?:years|year|yrs|yr)',
        r'(\d+)\+?\s*(?:years|year|yrs|yr)\s+(?:in|with|using|working)',
        r'experience\s+(?:of\s+)?(\d+)\+?\s*(?:years|year|yrs|yr)'
    ]

    extracted_years = []

    for pattern in experience_patterns:
        matches = re.findall(
            pattern,
            job_clean
        )

        for match in matches:
            if isinstance(match, tuple):
                numbers = [
                    int(value)
                    for value in match
                    if str(value).isdigit()
                ]

                if len(numbers) > 0:
                    extracted_years.append(
                        max(numbers)
                    )
            else:
                if str(match).isdigit():
                    extracted_years.append(
                        int(match)
                    )

    if len(extracted_years) == 0:
        if any(
            keyword in job_clean
            for keyword in ["fresher", "fresh graduate", "entry level", "entry-level", "intern"]
        ):
            return 0

        return 0

    return max(extracted_years)

Cell 58 (RUN)

In [28]:
def normalize_skill_list(skills):
    normalization_map = {
        "nodejs": "node",
        "node js": "node",
        "microservice architecture": "microservices",
        "test driven development": "tdd",
        "ubuntu": "linux",
        "debian": "linux",
        "reactjs": "react",
        "react js": "react",
        "react-native": "react native"
    }

    normalized = []

    for skill in skills:
        normalized_skill = normalization_map.get(skill, skill)
        normalized.append(normalized_skill)

    return sorted(list(set(normalized)))

Cell 59 (RUN)

In [29]:
def suggest_job_match_improvements(
    resume_skills,
    job_skills,
    missing_skills,
    required_years,
    candidate_years,
    final_score
):
    suggestions = []

    resume_skills = set(resume_skills)
    job_skills = set(job_skills)
    missing_skills = sorted(list(set(missing_skills)))

    matched_skills = sorted(
        list(job_skills.intersection(resume_skills))
    )

    if required_years > candidate_years:
        suggestions.append(
            f"The job description appears to require about {required_years}+ years of experience, "
            f"while the CV shows about {candidate_years} years of formal experience. "
            "Add relevant internships, freelance work, part-time roles, academic projects, client work, or practical experience if available."
        )

    if len(missing_skills) > 0:
        suggestions.append(
            "Add or highlight these job-specific skills if you have them: "
            + ", ".join(missing_skills[:12])
        )

    if len(matched_skills) > 0:
        suggestions.append(
            "The CV already matches these job-relevant skills: "
            + ", ".join(matched_skills[:12])
        )

    if len(job_skills) > 0:
        skill_coverage = len(matched_skills) / len(job_skills)
    else:
        skill_coverage = 0

    if skill_coverage < 0.30:
        suggestions.append(
            "The CV has low keyword and skill overlap with this job description. "
            "Rewrite the summary, skills section, and project descriptions using relevant terms from the job description."
        )

    elif skill_coverage < 0.60:
        suggestions.append(
            "The CV has partial skill overlap. Improve alignment by adding more role-specific tools, responsibilities, and measurable outcomes."
        )

    else:
        suggestions.append(
            "The CV has good skill overlap with the job description. Strengthen it further with achievements, metrics, and role-specific examples."
        )

    if final_score < 0.50:
        suggestions.append(
            "Overall job alignment is weak. Consider applying to roles closer to the CV's current skills, or customize the CV more strongly for this job."
        )

    elif final_score < 0.75:
        suggestions.append(
            "Overall job alignment is moderate. Tailor the CV by emphasizing the most relevant experience, skills, projects, and achievements."
        )

    else:
        suggestions.append(
            "Overall job alignment is strong. The CV is well matched to the uploaded job description."
        )

    return suggestions

Cell 60: Tf-IDF Baseline (RUN)

In [30]:
def evaluate_with_tfidf_baseline(
    resume_clean,
    job_clean,
    resume_skills,
    job_skills,
    experience_match_score=1
):
    vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        max_features=5000
    )

    tfidf_matrix = vectorizer.fit_transform(
        [resume_clean, job_clean]
    )

    similarity_score = cosine_similarity(
        tfidf_matrix[0:1],
        tfidf_matrix[1:2]
    )[0][0]

    skill_score = calculate_skill_score(
        resume_skills,
        job_skills
    )

    final_score = (
        0.50 * similarity_score
        + 0.30 * skill_score
        + 0.20 * experience_match_score
    )

    label = classify(final_score)

    return {
        "model": "TF-IDF Baseline",
        "similarity_score": round(float(similarity_score), 4),
        "skill_score": round(float(skill_score), 4),
        "experience_match_score": round(float(experience_match_score), 4),
        "final_score": round(float(final_score), 4),
        "cv_category": label
    }

Cell 61 (RUN)

In [31]:
def compare_three_models_for_cv_and_job_text(cv_text, job_text):
    resume_clean = clean_text(cv_text)
    job_clean = clean_text(job_text)

    resume_skills = normalize_skill_list(
        extract_skills_from_text(resume_clean)
    )

    job_skills = normalize_skill_list(
        extract_skills_from_text(job_clean)
    )

    cv_sections = extract_sections(cv_text)

    candidate_experience = extract_experience_features(
        cv_text,
        cv_sections
    )

    candidate_years = candidate_experience["estimated_years_experience"]

    required_years = extract_required_experience_years(job_text)

    if required_years > 0:
        experience_match_score = min(candidate_years / required_years, 1)
    else:
        experience_match_score = 1

    expanded_resume_skills = normalize_skill_list(
        expand_skills(resume_skills)
    )

    expanded_job_skills = normalize_skill_list(
        expand_skills(job_skills)
    )

    matched_skills = sorted(
        list(
            set(expanded_resume_skills)
            .intersection(set(expanded_job_skills))
        )
    )

    missing_skills = sorted(
        list(
            set(expanded_job_skills)
            - set(expanded_resume_skills)
        )
    )

    models = [
        ("Sentence-BERT", bert_model),
        ("RoBERTa", roberta_model),
        ("DistilRoBERTa", distilroberta_model)
    ]

    results = []

    for model_name, model in models:
        model_result = evaluate_with_single_model(
            model_name,
            model,
            resume_clean,
            job_clean,
            resume_skills,
            job_skills,
            experience_match_score
        )

        results.append(model_result)

    tfidf_result = evaluate_with_tfidf_baseline(
        resume_clean,
        job_clean,
        resume_skills,
        job_skills,
        experience_match_score
    )

    results.append(tfidf_result)

    comparison_df = pd.DataFrame(results)

    best_model_row = comparison_df.loc[
        comparison_df["final_score"].idxmax()
    ]

    job_suggestions = suggest_job_match_improvements(
        resume_skills=expanded_resume_skills,
        job_skills=expanded_job_skills,
        missing_skills=missing_skills,
        required_years=required_years,
        candidate_years=candidate_years,
        final_score=best_model_row["final_score"]
    )

    return {
        "model_comparison": comparison_df,
        "best_model": best_model_row["model"],
        "best_job_match_score": float(best_model_row["final_score"]),
        "best_job_match_category": best_model_row["cv_category"],
        "required_years_experience": required_years,
        "candidate_years_experience": candidate_years,
        "experience_match_score": round(float(experience_match_score), 4),
        "resume_skills": resume_skills,
        "job_required_skills": job_skills,
        "matched_skills": matched_skills,
        "missing_skills": missing_skills,
        "job_suggestions": job_suggestions
    }

Cell 62 (RUN)

In [32]:
def evaluate_with_single_model(
    model_name,
    model,
    resume_clean,
    job_clean,
    resume_skills,
    job_skills,
    experience_match_score=1
):
    resume_embedding = model.encode([resume_clean])
    job_embedding = model.encode([job_clean])

    similarity_score = cosine_similarity(
        resume_embedding,
        job_embedding
    )[0][0]

    skill_score = calculate_skill_score(
        resume_skills,
        job_skills
    )

    final_score = (
        0.50 * similarity_score
        + 0.30 * skill_score
        + 0.20 * experience_match_score
    )

    label = classify(final_score)

    return {
        "model": model_name,
        "similarity_score": round(float(similarity_score), 4),
        "skill_score": round(float(skill_score), 4),
        "experience_match_score": round(float(experience_match_score), 4),
        "final_score": round(float(final_score), 4),
        "cv_category": label
    }

Cell 63 (RUN)

In [33]:
def evaluate_cv_system(
    resume_pdf_path=None,
    resume_text=None,
    job_pdf_path=None,
    job_text=None
):
    cv_text_quality = None
    job_text_quality = None

    if resume_text is not None and len(resume_text.strip()) > 0:
        cv_text = resume_text
        cv_text_quality = check_extracted_text_quality(cv_text)
    elif resume_pdf_path is not None:
        cv_text, cv_text_quality = get_text_from_pdf_with_quality(
            resume_pdf_path
        )
    else:
        cv_text = ""

    if len(cv_text.strip()) == 0:
        return {
            "error": "No CV text found. Please upload a resume PDF or paste CV text."
        }

    cv_result = evaluate_cv_without_job(cv_text)

    final_output = {
        "rubric_score": cv_result["rubric_score"],
        "cv_category": cv_result["cv_category"],
        "score_breakdown": cv_result["score_breakdown"],
        "structured_features": cv_result["structured_features"],
        "rubric_suggestions": cv_result["suggestions"],
        "cv_text_extraction_quality": cv_text_quality
    }

    job_text_final = ""

    if job_pdf_path is not None:
        job_text_final, job_text_quality = get_text_from_pdf_with_quality(
            job_pdf_path
        )

    elif job_text is not None and len(job_text.strip()) > 0:
        job_text_final = job_text
        job_text_quality = check_extracted_text_quality(job_text_final)

    if len(job_text_final.strip()) > 0:
        job_match_result = compare_three_models_for_cv_and_job_text(
            cv_text,
            job_text_final
        )

        best_job_match_score_100 = (
            job_match_result["best_job_match_score"] * 100
        )

        combined_score = (
            0.60 * cv_result["rubric_score"]
            + 0.40 * best_job_match_score_100
        )

        final_output["job_matching"] = job_match_result
        final_output["job_text_extraction_quality"] = job_text_quality
        final_output["combined_score"] = round(float(combined_score), 2)
        final_output["combined_category"] = classify_cv_from_100(
            combined_score
        )

    return final_output

Cell 64 (RUN)

In [34]:
result = evaluate_cv_system(
    resume_pdf_path=resume_pdf_path
)

result

{'rubric_score': 77,
 'cv_category': 'Strong',
 'score_breakdown': {'academic_record': 15,
  'work_experience': 7,
  'project_experience': 15,
  'technical_skills': 20,
  'soft_skills': 10,
  'language_proficiency': 0,
  'certifications': 0,
  'achievements_leadership': 10,
  'publications': 0},
 'structured_features': {'technical_skills': ['android',
   'android studio',
   'api',
   'artificial intelligence',
   'automated testing',
   'automation testing',
   'backend',
   'c',
   'c#',
   'c++',
   'cnn',
   'dart',
   'data preprocessing',
   'decision tree',
   'deep learning',
   'efficientnet',
   'firebase',
   'flutter',
   'git',
   'github',
   'google colab',
   'gradio',
   'logistic regression',
   'machine learning',
   'manual testing',
   'mobilenet',
   'model evaluation',
   'mysql',
   'numpy',
   'nunit',
   'pandas',
   'php',
   'postman',
   'pytest',
   'python',
   'random forest',
   'regression',
   'regression testing',
   'rest api',
   'selenium',
   'so

Cell 65 (RUN)

In [35]:
from sentence_transformers import SentenceTransformer

bert_model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

roberta_model = SentenceTransformer(
    'sentence-transformers/all-roberta-large-v1'
)

distilroberta_model = SentenceTransformer(
    'sentence-transformers/all-distilroberta-v1'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Cell 66 (RUN)

In [36]:
def expand_skills(skills):
    skills = set(skills)

    if "selenium" in skills or "pytest" in skills or "nunit" in skills:
        skills.add("automated testing")
        skills.add("test automation")
        skills.add("software testing")

    if "test automation" in skills or "automated testing" in skills:
        skills.add("quality assurance")
        skills.add("software quality")

    if "rest api" in skills:
        skills.add("api")

    if "api" in skills:
        skills.add("rest api")

    if "communication" in skills or "collaboration" in skills:
        skills.add("teamwork")

    return sorted(list(skills))

Cell 67 (RUN)

In [37]:
def calculate_skill_score(resume_skills, job_skills):
    resume_skills = set(expand_skills(resume_skills))
    job_skills = set(expand_skills(job_skills))

    if len(job_skills) == 0:
        return 0

    matched_skills = resume_skills.intersection(job_skills)

    return len(matched_skills) / len(job_skills)

Cell 68 (RUN)

In [38]:
def suggest_improvements_v2(resume_skills, job_skills, final_score):
    suggestions = []

    resume_skills_expanded = set(expand_skills(resume_skills))
    job_skills_expanded = set(expand_skills(job_skills))

    missing_skills = sorted(
        list(job_skills_expanded - resume_skills_expanded)
    )

    if missing_skills:
        suggestions.append(
            "Add or highlight these job-relevant skills: "
            + ", ".join(missing_skills[:10])
        )

    if "qa" in job_skills_expanded or "quality assurance" in job_skills_expanded:
        if "manual testing" not in resume_skills_expanded:
            suggestions.append(
                "Mention manual testing experience or academic testing practice."
            )

        if "bug reporting" not in resume_skills_expanded:
            suggestions.append(
                "Add bug reporting, defect tracking, or test case writing experience."
            )

        if "selenium" in resume_skills_expanded:
            suggestions.append(
                "Your Selenium automation experience is relevant. Highlight it more clearly for QA roles."
            )

    if final_score < 0.50:
        suggestions.append(
            "Rewrite the career objective for the target role and emphasize the most relevant skills from the job description."
        )

    elif final_score < 0.75:
        suggestions.append(
            "Improve alignment by adding job-specific keywords, measurable project outcomes, and role-focused achievements."
        )

    else:
        suggestions.append(
            "The CV is well aligned with the uploaded job description."
        )

    return suggestions

Cell 69 (RUN)

In [39]:
def classify(score):
    if score >= 0.75:
        return "Strong"
    elif score >= 0.50:
        return "Average"
    else:
        return "Weak"

Cell 70 (RUN)

In [40]:
def get_category_status(score, max_score):
    percentage = (score / max_score) * 100

    if percentage >= 75:
        return "Strong"
    elif percentage >= 50:
        return "Average"
    else:
        return "Weak"

Cell 71 (RUN)

In [41]:
def create_ats_score(final_result):
    score = final_result["rubric_score"]

    if score >= 75:
        status = "Strong ATS compatibility"
    elif score >= 50:
        status = "Moderate ATS compatibility"
    else:
        status = "Weak ATS compatibility"

    ats_result = {
        "ats_score": score,
        "ats_status": status,
        "ats_category": final_result["cv_category"]
    }

    return ats_result

Cell 72 (RUN)

In [42]:
def create_nine_category_breakdown(final_result):
    breakdown = final_result["score_breakdown"]

    max_scores = {
        "academic_record": 15,
        "work_experience": 15,
        "project_experience": 15,
        "technical_skills": 20,
        "soft_skills": 10,
        "language_proficiency": 5,
        "certifications": 5,
        "achievements_leadership": 10,
        "publications": 5
    }

    rows = []

    for category, score in breakdown.items():
        max_score = max_scores.get(category, 10)

        rows.append({
            "Category": category.replace("_", " ").title(),
            "Score": score,
            "Max Score": max_score,
            "Percentage": round((score / max_score) * 100, 2),
            "Status": get_category_status(score, max_score)
        })

    return pd.DataFrame(rows)

Cell 73 (RUN)

In [43]:
def create_skills_analysis(final_result):
    features = final_result["structured_features"]

    technical_skills = features["technical_skills"]
    soft_skills = features["soft_skills"]

    skills_analysis = {
        "technical_skill_count": len(technical_skills),
        "soft_skill_count": len(soft_skills),
        "technical_skills": technical_skills,
        "soft_skills": soft_skills,
        "keywords": features.get("keywords", {})
    }

    if "job_matching" in final_result:
        job_match = final_result["job_matching"]

        skills_analysis["matched_skills"] = job_match["matched_skills"]
        skills_analysis["missing_skills"] = job_match["missing_skills"]
        skills_analysis["job_required_skills"] = job_match["job_required_skills"]

    return skills_analysis

Cell 74 (RUN)

In [44]:
def create_final_hiring_score(final_result):
    if "combined_score" not in final_result:
        return {
            "hiring_score": final_result["rubric_score"],
            "hiring_category": final_result["cv_category"],
            "note": "No job description was provided, so hiring score is based only on CV rubric score."
        }

    return {
        "hiring_score": final_result["combined_score"],
        "hiring_category": final_result["combined_category"],
        "cv_quality_score": final_result["rubric_score"],
        "job_match_score": round(
            final_result["job_matching"]["best_job_match_score"] * 100,
            2
        ),
        "best_model": final_result["job_matching"]["best_model"]
    }

Cell 75 (RUN)

In [45]:
def create_skill_gap_analysis(final_result):
    if "job_matching" not in final_result:
        return {
            "message": "No job description was provided.",
            "matched_skills": [],
            "missing_skills": []
        }

    job_match = final_result["job_matching"]

    matched_skills = job_match["matched_skills"]
    missing_skills = job_match["missing_skills"]
    job_required_skills = job_match["job_required_skills"]

    if len(job_required_skills) > 0:
        gap_percentage = round(
            (len(missing_skills) / len(job_required_skills)) * 100,
            2
        )
        match_percentage = round(
            (len(matched_skills) / len(job_required_skills)) * 100,
            2
        )
    else:
        gap_percentage = 0
        match_percentage = 0

    return {
        "matched_skills": matched_skills,
        "missing_skills": missing_skills,
        "match_percentage": match_percentage,
        "gap_percentage": gap_percentage
    }

Cell 76 (RUN)

In [46]:
def create_learning_roadmap(final_result):
    if "job_matching" not in final_result:
        return [
            "Upload a job description to generate a job-specific learning roadmap."
        ]

    missing_skills = final_result["job_matching"]["missing_skills"]

    roadmap = []

    for skill in missing_skills[:10]:
        roadmap.append(
            f"Learn or improve {skill}: study the basics, build a small project, and add it to your CV if relevant."
        )

    if len(roadmap) == 0:
        roadmap.append(
            "No major missing skills detected. Improve your CV by adding measurable outcomes and stronger project descriptions."
        )

    return roadmap

Cell 77 (RUN)

In [47]:
def create_model_dashboard(final_result):
    if "job_matching" not in final_result:
        return pd.DataFrame()

    model_df = final_result["job_matching"]["model_comparison"].copy()

    best_model = final_result["job_matching"]["best_model"]

    model_df["is_best_model"] = model_df["model"].apply(
        lambda model: "Yes" if model == best_model else "No"
    )

    return model_df

Cell 78 (RUN)

In [48]:
def create_explainable_ai_summary(final_result):
    explanation = []

    explanation.append(
        f"The CV received a rubric score of {final_result['rubric_score']}/100, "
        f"which is categorized as {final_result['cv_category']}."
    )

    strongest_categories = []
    weakest_categories = []

    max_scores = {
        "academic_record": 15,
        "work_experience": 15,
        "project_experience": 15,
        "technical_skills": 20,
        "soft_skills": 10,
        "language_proficiency": 5,
        "certifications": 5,
        "achievements_leadership": 10,
        "publications": 5
    }

    for category, score in final_result["score_breakdown"].items():
        max_score = max_scores.get(category, 10)
        percentage = (score / max_score) * 100

        if percentage >= 75:
            strongest_categories.append(category.replace("_", " "))
        elif percentage < 50:
            weakest_categories.append(category.replace("_", " "))

    if strongest_categories:
        explanation.append(
            "Strong areas: " + ", ".join(strongest_categories) + "."
        )

    if weakest_categories:
        explanation.append(
            "Weak or missing areas: " + ", ".join(weakest_categories) + "."
        )

    if "job_matching" in final_result:
        job_match = final_result["job_matching"]

        explanation.append(
            f"For the uploaded job description, the best model was {job_match['best_model']} "
            f"with a job match score of {round(job_match['best_job_match_score'] * 100, 2)}/100."
        )

        explanation.append(
            f"The candidate matched {len(job_match['matched_skills'])} job skills "
            f"and missed {len(job_match['missing_skills'])} job skills."
        )

        explanation.append(
            f"The job requires about {job_match['required_years_experience']} years of experience, "
            f"while the CV shows {job_match['candidate_years_experience']} years."
        )

    return "\n\n".join(explanation)

Cell 79 (RUN)

In [49]:
def explain_best_model_selection(final_result):
    if "job_matching" not in final_result:
        return (
            "No job description was provided, so model comparison was not performed."
        )

    model_df = final_result["job_matching"]["model_comparison"].copy()

    best_model = final_result["job_matching"]["best_model"]

    best_row = model_df[
        model_df["model"] == best_model
    ].iloc[0]

    explanation = []

    explanation.append(
        f"The best model for this CV-job matching task is {best_model}."
    )

    explanation.append(
        f"{best_model} achieved the highest final job match score: "
        f"{round(float(best_row['final_score']) * 100, 2)}/100."
    )

    explanation.append(
        "The final score is calculated using semantic or lexical similarity, "
        "skill overlap, and experience match score."
    )

    explanation.append(
        f"For {best_model}, the similarity score was "
        f"{round(float(best_row['similarity_score']) * 100, 2)}/100, "
        f"the skill score was {round(float(best_row['skill_score']) * 100, 2)}/100, "
        f"and the experience match score was "
        f"{round(float(best_row['experience_match_score']) * 100, 2)}/100."
    )

    other_models = model_df[
        model_df["model"] != best_model
    ]

    for _, row in other_models.iterrows():
        explanation.append(
            f"{row['model']} was not selected because its final score was "
            f"{round(float(row['final_score']) * 100, 2)}/100, "
            f"which is lower than {best_model}'s score."
        )

    explanation.append(
        "Sentence-BERT, RoBERTa, and DistilRoBERTa use dense sentence embeddings "
        "to capture semantic meaning between the CV and job description."
    )

    explanation.append(
        "TF-IDF Baseline is a traditional keyword-based model. It is useful as a baseline "
        "because it measures exact term and phrase overlap, but it may miss semantic similarity "
        "when different words express similar meaning."
    )

    explanation.append(
        "The skill score and experience score are rule-based, so they remain the same across models. "
        "The main difference between models comes from the similarity score."
    )

    return "\n\n".join(explanation)

Cell 80 (RUN)

In [50]:
def create_extra_project_features(final_result):
    extra_features = {
        "ats_score": create_ats_score(final_result),
        "category_breakdown": create_nine_category_breakdown(final_result),
        "skills_analysis": create_skills_analysis(final_result),
        "hiring_score": create_final_hiring_score(final_result),
        "skill_gap_analysis": create_skill_gap_analysis(final_result),
        "learning_roadmap": create_learning_roadmap(final_result),
        "model_dashboard": create_model_dashboard(final_result),
        "explainable_ai_summary": create_explainable_ai_summary(final_result),
        "best_model_explanation": explain_best_model_selection(final_result)
    }

    return extra_features

Cell 81 (RUN)

In [51]:
final_result = evaluate_cv_system(
    resume_pdf_path=resume_pdf_path,
    job_pdf_path=job_pdf_path
)

extra_features = create_extra_project_features(final_result)

extra_features

{'ats_score': {'ats_score': 77,
  'ats_status': 'Strong ATS compatibility',
  'ats_category': 'Strong'},
 'category_breakdown':                   Category  Score  Max Score  Percentage  Status
 0          Academic Record     15         15      100.00  Strong
 1          Work Experience      7         15       46.67    Weak
 2       Project Experience     15         15      100.00  Strong
 3         Technical Skills     20         20      100.00  Strong
 4              Soft Skills     10         10      100.00  Strong
 5     Language Proficiency      0          5        0.00    Weak
 6           Certifications      0          5        0.00    Weak
 7  Achievements Leadership     10         10      100.00  Strong
 8             Publications      0          5        0.00    Weak,
 'skills_analysis': {'technical_skill_count': 47,
  'soft_skill_count': 5,
  'technical_skills': ['android',
   'android studio',
   'api',
   'artificial intelligence',
   'automated testing',
   'automation tes

Cell 82 (RUN)

In [52]:
!pip install gradio

Cell 83 (RUN)

In [53]:
import gradio as gr
import pandas as pd

Cell 84 (RUN)

In [54]:
def safe_file_path(file_obj):
    if file_obj is None:
        return None

    if isinstance(file_obj, str):
        return file_obj

    return file_obj.name

Cell 85 (RUN)

In [55]:
def format_list(items, empty_text="Not detected"):
    if items is None or len(items) == 0:
        return empty_text

    return ", ".join(items)

Cell 86 (RUN)

In [56]:
def list_to_html(items):
    if items is None or len(items) == 0:
        return "<p style='color:#64748b;'>Not detected</p>"

    html = "<ul>"

    for item in items:
        html += f"<li>{item}</li>"

    html += "</ul>"

    return html

Cell 87 (RUN)

In [57]:
def score_badge(category):
    color_map = {
        "Strong": "#16a34a",
        "Average": "#ca8a04",
        "Weak": "#dc2626"
    }

    color = color_map.get(category, "#334155")

    return f"""
    <span style="
        background:{color};
        color:white;
        padding:6px 12px;
        border-radius:999px;
        font-weight:600;
        font-size:14px;
    ">
        {category}
    </span>
    """

Cell 88 (RUN)

In [58]:
def create_ats_html(extra_features):
    ats = extra_features["ats_score"]

    return f"""
    <div class="result-card">
        <h2>ATS Compatibility Score</h2>

        <div class="score-main">
            {ats["ats_score"]}/100
        </div>

        <div style="margin-top:12px;">
            {score_badge(ats["ats_category"])}
        </div>

        <p style="margin-top:18px; color:#475569;">
            {ats["ats_status"]}
        </p>
    </div>
    """

Cell 89 (RUN)

In [59]:
def create_hiring_html(extra_features):
    hiring = extra_features["hiring_score"]

    html = f"""
    <div class="result-card">
        <h2>Final Hiring / Ranking Score</h2>

        <div class="score-main">
            {hiring["hiring_score"]}/100
        </div>

        <div style="margin-top:12px;">
            {score_badge(hiring["hiring_category"])}
        </div>
    """

    if "job_match_score" in hiring:
        html += f"""
        <hr>
        <p><b>CV Quality Score:</b> {hiring["cv_quality_score"]}/100</p>
        <p><b>Job Match Score:</b> {hiring["job_match_score"]}/100</p>
        <p><b>Best Matching Model:</b> {hiring["best_model"]}</p>
        """
    else:
        html += f"""
        <p style="color:#64748b;">{hiring["note"]}</p>
        """

    html += "</div>"

    return html

Cell 90 (RUN)

In [60]:
def create_skills_html(extra_features):
    skills = extra_features["skills_analysis"]

    html = f"""
    <div class="result-card">
        <h2>Skills and Keyword Analysis</h2>

        <h3>Technical Skills ({skills["technical_skill_count"]})</h3>
        {list_to_html(skills["technical_skills"])}

        <h3>Soft Skills ({skills["soft_skill_count"]})</h3>
        {list_to_html(skills["soft_skills"])}
    """

    if "keywords" in skills:
        keywords = skills["keywords"]

        html += f"""
        <hr>

        <h3>Known CV Keywords ({len(keywords.get("known_keywords", []))})</h3>
        {list_to_html(keywords.get("known_keywords", []))}

        <h3>Frequent Keywords / Phrases</h3>
        {list_to_html(keywords.get("frequent_keywords", []))}
        """

    if "matched_skills" in skills:
        html += f"""
        <hr>

        <h3>Matched Job Skills</h3>
        {list_to_html(skills["matched_skills"])}

        <h3>Missing Job Skills</h3>
        {list_to_html(skills["missing_skills"])}
        """

    html += "</div>"

    return html

Cell 91 (RUN)

In [61]:
def create_gap_html(extra_features):
    gap = extra_features["skill_gap_analysis"]

    if "message" in gap:
        return f"""
        <div class="result-card">
            <h2>Skill Gap Analysis</h2>
            <p>{gap["message"]}</p>
        </div>
        """

    return f"""
    <div class="result-card">
        <h2>Skill Gap Analysis</h2>

        <p><b>Skill Match Percentage:</b> {gap["match_percentage"]}%</p>
        <p><b>Skill Gap Percentage:</b> {gap["gap_percentage"]}%</p>

        <h3>Matched Skills</h3>
        {list_to_html(gap["matched_skills"])}

        <h3>Missing Skills</h3>
        {list_to_html(gap["missing_skills"])}
    </div>
    """

Cell 92 (RUN)

In [62]:
def create_roadmap_markdown(extra_features):
    roadmap = extra_features["learning_roadmap"]

    markdown = "# Learning Roadmap\n\n"

    for idx, item in enumerate(roadmap, start=1):
        markdown += f"{idx}. {item}\n"

    return markdown

Cell 93 (RUN)

In [63]:
def create_summary_markdown_v2(final_result, extra_features):
    ats = extra_features["ats_score"]
    hiring = extra_features["hiring_score"]

    markdown = f"""
# CV Evaluation Summary

## CV Quality

**Rubric Score:** {final_result["rubric_score"]}/100\n
**CV Category:** {final_result["cv_category"]}\n
**ATS Status:** {ats["ats_status"]}\n

## Final Ranking

**Final Score:** {hiring["hiring_score"]}/100\n
**Final Category:** {hiring["hiring_category"]}\n
"""

    if "cv_text_extraction_quality" in final_result:
        quality = final_result["cv_text_extraction_quality"]

        markdown += f"""

## CV Text Extraction Quality

**Status:** {quality["status"]}\n
**Word Count:** {quality["word_count"]}\n
**Total Pages:** {quality.get("total_pages", "N/A")}\n
**Text Pages:** {quality.get("text_pages", "N/A")}\n
**OCR Pages:** {quality.get("ocr_pages", "N/A")}\n
**Failed Pages:** {quality.get("failed_pages", "N/A")}\n
**Note:** {quality["message"]}\n
"""

    if "job_text_extraction_quality" in final_result:
        quality = final_result["job_text_extraction_quality"]

        markdown += f"""

## Job Description Text Extraction Quality

**Status:** {quality["status"]}\n
**Word Count:** {quality["word_count"]}\n
**Total Pages:** {quality.get("total_pages", "N/A")}\n
**Text Pages:** {quality.get("text_pages", "N/A")}\n
**OCR Pages:** {quality.get("ocr_pages", "N/A")}\n
**Failed Pages:** {quality.get("failed_pages", "N/A")}\n
**Note:** {quality["message"]}\n
"""

    return markdown

Cell 94 (RUN)

In [64]:
def create_recommendations_markdown_v2(final_result):
    markdown = "# Recommendations\n\n"

    markdown += "## CV Improvement Suggestions\n\n"

    for suggestion in final_result["rubric_suggestions"]:
        markdown += f"- {suggestion}\n"

    if "job_matching" in final_result:
        markdown += "\n## Job-Specific Suggestions\n\n"

        job_suggestions = final_result["job_matching"].get(
            "job_suggestions",
            []
        )

        if len(job_suggestions) == 0:
            markdown += "- No job-specific suggestions available.\n"
        else:
            for suggestion in job_suggestions:
                markdown += f"- {suggestion}\n"

    return markdown

Cell 95 (RUN)

In [65]:
def create_job_match_markdown_v2(final_result):
    if "job_matching" not in final_result:
        return "# Job Match\n\nNo job description was provided."

    job_match = final_result["job_matching"]

    markdown = f"""
# Job Match Details

## Best Model

**Model:** {job_match["best_model"]}\n
**Job Match Score:** {round(job_match["best_job_match_score"] * 100, 2)}/100\n
**Category:** {job_match["best_job_match_category"]}\n

## Experience

**Required Experience:** {job_match["required_years_experience"]}+ years\n
**Candidate Experience:** {job_match["candidate_years_experience"]} years\n
**Experience Match Score:** {job_match["experience_match_score"]}\n

## Matched Skills

{", ".join(job_match["matched_skills"]) if job_match["matched_skills"] else "None"}\n

## Missing Skills

{", ".join(job_match["missing_skills"]) if job_match["missing_skills"] else "None"}\n
"""

    return markdown

Cell 96 (RUN)

In [66]:
cv_text, cv_quality = get_text_from_pdf_with_quality(resume_pdf_path)

print(cv_quality)
print(cv_text[:1000])

{'status': 'Good text quality', 'word_count': 434, 'message': 'Text extraction appears sufficient.', 'total_pages': 2, 'text_pages': 2, 'ocr_pages': 0, 'failed_pages': 0}
        Kazi Nusrat Sinthia 
                📍 31/1, West Monipur, Mirpur-2, Dhaka-1216 
                  📞 01772220850    📧 nusratsinthia11628@gmail.com 
                   LinkedIn: https://www.linkedin.com/in/kazi-nusrat-sinthia-3b280029a/   
                   Github: https://github.com/Sinthia-11628/ 
 
 
Career Objective 
Motivated Computer Science and Engineering student seeking a full-time SQA Intern position. 
Familiar with software testing fundamentals, SDLC, STLC, manual testing, test-case design. Eager 
to contribute strong analytical skills and hands-on experience with Python, Postman, and Git to 
real-world software-quality projects. Available for immediate joining. 
Academic Qualifications: 
❖ Bachelor of Science in Computer Science and Engineering 
Jagannath University | Expected Graduation: 2026 
❖ H

Cell 97 (RUN)

In [67]:
job_text_extracted, job_quality = get_text_from_pdf_with_quality(job_pdf_path)

print(job_quality)
print(job_text_extracted[:1000])

{'status': 'Good text quality', 'word_count': 168, 'message': 'Text extraction appears sufficient.', 'total_pages': 1, 'text_pages': 1, 'ocr_pages': 0, 'failed_pages': 0}
Software Quality Assurance Engineer (SQA) 
We are expanding our engineering team at Polygon Technology! If building 
high-performance systems and scaling robust architectures is what you do best, we want 
you. 
 
We are currently looking for experienced professionals for Multiple Roles: 
 Software Engineer (Golang) 
 Software Engineer (Laravel / PHP) 
 Software Quality Assurance Engineer 
 Apply now: https://shorturl.at/URT6z 
Key Responsibilities 
Test Planning and Strategy 
Analyze requirements, user stories, and technical specifications to identify test scenarios 
and edge cases. 
Prepare and maintain detailed test plans, test cases, and test checklists for new and 
existing features. 
Manual Testing 
Execute functional, regression, integration, and system tests for web and mobile 
applications. 
Validate UI, UX, a

Cell 98 (RUN)

In [68]:
def run_cv_evaluator_gradio(resume_pdf, job_pdf):
    try:
        resume_pdf_path = safe_file_path(resume_pdf)
        job_pdf_path = safe_file_path(job_pdf)

        if resume_pdf_path is None:
            empty_df = pd.DataFrame()
            return (
                "Please upload a resume PDF.",
                "",
                empty_df,
                "",
                "",
                "",
                "",
                "",
                "",
                empty_df,
                "",
                "No resume uploaded."
            )

        if job_pdf_path is not None:
            final_result = evaluate_cv_system(
                resume_pdf_path=resume_pdf_path,
                job_pdf_path=job_pdf_path
            )
        else:
            final_result = evaluate_cv_system(
                resume_pdf_path=resume_pdf_path
            )

        extra_features = create_extra_project_features(final_result)

        summary_output = create_summary_markdown_v2(
            final_result,
            extra_features
        )

        ats_output = create_ats_html(extra_features)

        scores_output = extra_features["category_breakdown"]

        skills_output = create_skills_html(extra_features)

        recommendations_output = create_recommendations_markdown_v2(
            final_result
        )

        job_match_output = create_job_match_markdown_v2(final_result)

        hiring_output = create_hiring_html(extra_features)

        gap_output = create_gap_html(extra_features)

        roadmap_output = create_roadmap_markdown(extra_features)

        model_output = extra_features["model_dashboard"]

        explain_output = extra_features["explainable_ai_summary"]

        status_output = "Evaluation completed successfully."

        return (
            summary_output,
            ats_output,
            scores_output,
            skills_output,
            recommendations_output,
            job_match_output,
            hiring_output,
            gap_output,
            roadmap_output,
            model_output,
            explain_output,
            status_output
        )

    except Exception as error:
        empty_df = pd.DataFrame()

        return (
            f"Error: {str(error)}",
            "",
            empty_df,
            "",
            "",
            "",
            "",
            "",
            "",
            empty_df,
            "",
            f"Error: {str(error)}"
        )

Cell 99 (RUN)

In [69]:
custom_css = """
.gradio-container {
    max-width: 1400px !important;
    margin: auto !important;
    font-family: Inter, Arial, sans-serif;
}

.header-container {
    text-align: center;
    padding: 30px;
    background: linear-gradient(135deg, #2563eb 0%, #0f172a 100%);
    color: white;
    border-radius: 12px;
    margin-bottom: 24px;
}

.result-card {
    border: 1px solid #e2e8f0;
    background:  #0f172a;
    border-radius: 12px;
    padding: 24px;
    box-shadow: 0 6px 18px rgba(15, 23, 42, 0.06);
}

.score-main {
    font-size: 48px;
    font-weight: 800;
    color: #2563eb;
    margin-top: 10px;
}

.placeholder {
    text-align: center;
    padding: 70px;
    color: #94a3b8;
}

footer {
    visibility: hidden;
}
"""

Cell 100 (RUN)

In [70]:
def clear_all_outputs():
    empty_html = """
    <div class="placeholder">
        <h3>No result yet</h3>
        <p>Upload files and click Evaluate CV.</p>
    </div>
    """

    return (
        None,
        None,
        "*Upload CV to see summary.*",
        empty_html,
        pd.DataFrame(),
        empty_html,
        "*Upload CV to see recommendations.*",
        "*Upload job description to see matching result.*",
        empty_html,
        empty_html,
        "*Upload job description to generate a learning roadmap.*",
        pd.DataFrame(),
        "*Upload CV to see explainability summary.*",
        ""
    )

Cell 101 (RUN)

Gradio shuru

In [74]:
with gr.Blocks(
    title="Automatic CV Evaluator and Job Matcher",
    theme=gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="slate",
        neutral_hue="slate"
    ),
    css=custom_css
) as demo:

    gr.HTML("""
    <div class="header-container">
        <h1>Automatic CV Evaluator and Job Matcher</h1>
        <p>
            Upload a CV and optional job description to generate rubric scoring,
            ATS analysis, skill gap analysis, 3-model matching, and improvement suggestions.
        </p>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("## Upload Documents")

            resume_input = gr.File(
                label="CV / Resume PDF",
                file_types=[".pdf"],
                type="filepath"
            )

            job_input = gr.File(
                label="Job Description PDF Optional",
                file_types=[".pdf"],
                type="filepath"
            )

            with gr.Row():
                evaluate_button = gr.Button(
                    "Evaluate CV",
                    variant="primary",
                    size="lg"
                )

                clear_button = gr.Button(
                    "Clear",
                    variant="secondary",
                    size="lg"
                )

            status_output = gr.Textbox(
                label="Processing Status",
                lines=3,
                interactive=False
            )

            gr.Markdown("""
            ## Included Features

            - ATS compatibility score
            - 9-category rubric breakdown
            - Technical and soft skills analysis
            - 3 transformer models + TF-IDF baseline matching
            - Skill gap analysis
            - Final hiring/ranking score
            - Learning roadmap
            - Model comparison dashboard
            - Explainable AI summary
            - Improvement recommendations
            """)

        with gr.Column(scale=2):
            gr.Markdown("## Analysis Results")

            with gr.Tabs():
                with gr.TabItem("Summary"):
                    summary_output = gr.Markdown(
                        value="*Upload CV to see summary.*"
                    )

                with gr.TabItem("ATS Score"):
                    ats_output = gr.HTML(
                        value="""
                        <div class="placeholder">
                            <h3>No ATS score yet</h3>
                            <p>Upload CV to calculate ATS score.</p>
                        </div>
                        """
                    )

                with gr.TabItem("Scores"):
                    scores_output = gr.Dataframe(
                        label="9-Category Rubric Breakdown",
                        interactive=False,
                        wrap=True
                    )

                with gr.TabItem("Skills"):
                    skills_output = gr.HTML(
                        value="""
                        <div class="placeholder">
                            <h3>No skills analysis yet</h3>
                            <p>Upload CV to analyze skills.</p>
                        </div>
                        """
                    )

                with gr.TabItem("Recommendations"):
                    recommendations_output = gr.Markdown(
                        value="*Upload CV to see recommendations.*"
                    )

                with gr.TabItem("Job Match"):
                    job_match_output = gr.Markdown(
                        value="*Upload job description to see matching result.*"
                    )

                with gr.TabItem("Hiring Score"):
                    hiring_output = gr.HTML(
                        value="""
                        <div class="placeholder">
                            <h3>No hiring score yet</h3>
                            <p>Upload job description to calculate final ranking.</p>
                        </div>
                        """
                    )

                with gr.TabItem("Skill Gap"):
                    gap_output = gr.HTML(
                        value="""
                        <div class="placeholder">
                            <h3>No skill gap analysis yet</h3>
                            <p>Upload job description for gap analysis.</p>
                        </div>
                        """
                    )

                with gr.TabItem("Learning Plan"):
                    roadmap_output = gr.Markdown(
                        value="*Upload job description to generate a learning roadmap.*"
                    )

                with gr.TabItem("Model Dashboard"):
                    model_output = gr.Dataframe(
                        label="Model Comparison",
                        interactive=False,
                        wrap=True
                    )

                with gr.TabItem("Explainability"):
                    explain_output = gr.Markdown(
                        value="*Upload CV to see explainability summary.*"
                    )

    evaluate_button.click(
        fn=run_cv_evaluator_gradio,
        inputs=[
            resume_input,
            job_input
        ],
        outputs=[
            summary_output,
            ats_output,
            scores_output,
            skills_output,
            recommendations_output,
            job_match_output,
            hiring_output,
            gap_output,
            roadmap_output,
            model_output,
            explain_output,
            status_output
        ]
    )

    clear_button.click(
        fn=clear_all_outputs,
        inputs=[],
        outputs=[
            resume_input,
            job_input,
            summary_output,
            ats_output,
            scores_output,
            skills_output,
            recommendations_output,
            job_match_output,
            hiring_output,
            gap_output,
            roadmap_output,
            model_output,
            explain_output,
            status_output
        ]
    )

/tmp/ipykernel_1188/3184755566.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Cell 102 (RUN)

In [ ]:
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fb230f1dbf44967ea0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
def explain_best_model_selection(final_result):
    if "job_matching" not in final_result:
        return (
            "No job description was provided, so model comparison was not performed."
        )

    model_df = final_result["job_matching"]["model_comparison"].copy()

    best_model = final_result["job_matching"]["best_model"]

    best_row = model_df[
        model_df["model"] == best_model
    ].iloc[0]

    explanation = []

    explanation.append(
        f"The best model for this CV-job matching task is {best_model}."
    )

    explanation.append(
        f"{best_model} achieved the highest final job match score: "
        f"{round(float(best_row['final_score']) * 100, 2)}/100."
    )

    explanation.append(
        "The final job match score is calculated using semantic similarity, "
        "skill overlap, and experience match score."
    )

    explanation.append(
        f"For {best_model}, the semantic similarity score was "
        f"{round(float(best_row['similarity_score']) * 100, 2)}/100, "
        f"the skill score was {round(float(best_row['skill_score']) * 100, 2)}/100, "
        f"and the experience match score was "
        f"{round(float(best_row['experience_match_score']) * 100, 2)}/100."
    )

    other_models = model_df[
        model_df["model"] != best_model
    ]

    for _, row in other_models.iterrows():
        explanation.append(
            f"{row['model']} was not selected because its final score was "
            f"{round(float(row['final_score']) * 100, 2)}/100, "
            f"which is lower than {best_model}'s score."
        )

    explanation.append(
        "The skill score and experience score are the same across models because "
        "they are rule-based. The main difference between models comes from the "
        "semantic similarity score generated by each embedding model."
    )

    explanation.append(
        "Therefore, the selected model is the one that best captured the semantic "
        "relationship between the uploaded CV and the uploaded job description."
    )

    return "\n\n".join(explanation)

In [ ]:
model_explanation = explain_best_model_selection(final_result)

print(model_explanation)

The best model for this CV-job matching task is Sentence-BERT.

Sentence-BERT achieved the highest final job match score: 76.03/100.

The final job match score is calculated using semantic similarity, skill overlap, and experience match score.

For Sentence-BERT, the semantic similarity score was 52.07/100, the skill score was 100.0/100, and the experience match score was 100.0/100.

RoBERTa was not selected because its final score was 72.55/100, which is lower than Sentence-BERT's score.

DistilRoBERTa was not selected because its final score was 67.26/100, which is lower than Sentence-BERT's score.

The skill score and experience score are the same across models because they are rule-based. The main difference between models comes from the semantic similarity score generated by each embedding model.

Therefore, the selected model is the one that best captured the semantic relationship between the uploaded CV and the uploaded job description.
